# iGEM IITB 2026 dry lab — current work

Everything in one notebook. It writes the code to disk itself, downloads the Yeast9 model, and runs all three analyses. **Runtime > Run all**, roughly 20 minutes.

## What this answers

1. **Feasibility** — at a viable growth rate, what C14:0 yield and purity can the model reach, and does any knockout improve it?
2. **Is the scoring trustworthy** — a pre-flight check on the scoring function itself.
3. **Is a learned search worth it** — reinforcement learning vs greedy vs brute force.

## The three headline results

- **Wild-type yeast already exports 100% pure C14:0** in this model, up to 90% of normal growth. No edits needed. No tested knockout improves on it; POX1 knockouts abolish production entirely.
- **The model makes C14:0 by chewing C16:0 down via beta-oxidation**, not by early release from synthesis. So it cannot represent the FAS2 strategy, and the degradation knockdown plan is inverted relative to what the model predicts.
- **The earlier profile-matching score was untestable** — it ranked knockouts by 0.01% differences, far below GC-MS resolution. The tool catches this and refuses to issue proposals.

**On probability:** FBA computes what is *possible* under stoichiometry and the bounds set. It is an upper limit, not a prediction and not a probability. It models no enzyme kinetics and no regulation. Nothing here is validated against wet-lab data.

## Step 1 — install and fetch the model

In [1]:
!pip install -q cobra pyyaml numpy pandas
!mkdir -p rl-strain-design/rl_strain_design rl-strain-design/configs
%cd rl-strain-design
import os
if not os.path.exists('yeast-GEM.xml'):
    !wget -q https://github.com/SysBioChalmers/yeast-GEM/raw/main/model/yeast-GEM.xml
print('model file:', os.path.getsize('yeast-GEM.xml')//1000, 'KB')

/tmp/claude-0/-home-claude/2ec11906-2834-5bce-ae47-da8a467dff5a/scratchpad/nb2/rl-strain-design
model file: 11568 KB


## Step 2 — write the package files
Nothing to edit here. One cell per source file.

In [2]:
%%writefile rl_strain_design/__init__.py
"""
rl_strain_design — RL-guided combinatorial gene-knockout search for
genome-scale metabolic models.

Given a target product-ratio profile and a COBRApy-compatible GEM, this
package trains a linear contextual-bandit agent to propose a knockout
combo that shifts flux toward that profile, without re-running brute-force
FBA over every combination every time someone asks for a new profile.

Quick start:

    from rl_strain_design import load_config, CobraEvaluator, train_agent, run_episode

    config = load_config("configs/yeast9_methyl_myristate.yaml")
    evaluator = CobraEvaluator(config)
    agent, history = train_agent(config, evaluator, n_episodes=500)

    target_profile = {"C14:0": 0.6, "C16:0": 0.4}
    combo, score, trace = run_episode(agent, evaluator, target_profile, train=False)
    print(combo, score)

See README.md for the full walkthrough and configs/yeast9_methyl_myristate.yaml
for a complete worked example (the project this package was extracted from).
"""

from .config import SearchConfig, load_config
from .filters import Rule, load_rules, is_valid_combo
from .agent import LinearQAgent, build_features, feature_dim
from .search import (
    run_episode, train_agent, brute_force_best, greedy_search,
    evaluate_agent_vs_bruteforce, sample_random_target_profile,
)
from .cobra_evaluator import CobraEvaluator, cosine_similarity, growth_penalty
from .diagnostics import (
    run_diagnostics, DiagnosticReport, CheckResult, SOLVER_NOISE_FLOOR,
)
from .proposals import build_proposal_sheet, ProposalSheet, Proposal
from .benchmark import (
    SyntheticLandscape, run_scaling_benchmark, benchmark_table,
    crossover_summary, combination_count,
)

__all__ = [
    "run_diagnostics", "DiagnosticReport", "CheckResult", "SOLVER_NOISE_FLOOR",
    "build_proposal_sheet", "ProposalSheet", "Proposal",
    "SyntheticLandscape", "run_scaling_benchmark", "benchmark_table",
    "crossover_summary", "combination_count",
    "SearchConfig", "load_config",
    "Rule", "load_rules", "is_valid_combo",
    "LinearQAgent", "build_features", "feature_dim",
    "run_episode", "train_agent", "brute_force_best", "greedy_search",
    "evaluate_agent_vs_bruteforce", "sample_random_target_profile",
    "CobraEvaluator", "cosine_similarity", "growth_penalty",
]

__version__ = "0.1.0"

Writing rl_strain_design/__init__.py


In [3]:
%%writefile rl_strain_design/config.py
"""
config.py — load a project config (genes, reactions, filters, model file)
=============================================================================
A SearchConfig is everything that's specific to ONE project/organism/target
family. The search engine (agent.py, search.py) and the generic evaluator
(cobra_evaluator.py) never hardcode biology -- they only ever read it from
here. To apply this package to a new organism or product, write a new YAML
file following configs/yeast9_methyl_myristate.yaml as a template; nothing
in rl_strain_design/ needs to change.
"""

from dataclasses import dataclass, field
import yaml

from .filters import load_rules, Rule


@dataclass
class SearchConfig:
    # --- model / organism ---
    model_path: str
    biomass_reaction_id: str

    # --- candidate gene space ---
    candidate_genes: list[str]
    gene_to_model_id: dict[str, str]   # e.g. common name -> systematic ORF ID
    max_knockout_order: int

    # --- target product(s) ---
    exchange_reactions: dict[str, str]  # e.g. "C14:0" -> "r_2193"

    # --- search-space pruning (see filters.py) ---
    filter_rules: list[Rule]

    # --- scoring / objective shape ---
    ratio_match_alpha: float = 0.5       # min fraction of max total output required
    hard_min_growth_fraction: float = 0.01
    growth_penalty_K: float = 0.5
    growth_penalty_n: float = 2.0

    # --- metadata (not used by the engine, just for provenance) ---
    name: str = "unnamed-project"
    description: str = ""
    sources: list[str] = field(default_factory=list)


def load_config(path: str) -> SearchConfig:
    with open(path) as f:
        raw = yaml.safe_load(f)

    required = ["model_path", "biomass_reaction_id", "candidate_genes",
                "gene_to_model_id", "exchange_reactions", "max_knockout_order"]
    missing = [k for k in required if k not in raw]
    if missing:
        raise ValueError(f"Config {path} is missing required keys: {missing}")

    return SearchConfig(
        model_path=raw["model_path"],
        biomass_reaction_id=raw["biomass_reaction_id"],
        candidate_genes=raw["candidate_genes"],
        gene_to_model_id=raw["gene_to_model_id"],
        max_knockout_order=raw["max_knockout_order"],
        exchange_reactions=raw["exchange_reactions"],
        filter_rules=load_rules(raw.get("pre_fba_filters", [])),
        ratio_match_alpha=raw.get("ratio_match_alpha", 0.5),
        hard_min_growth_fraction=raw.get("hard_min_growth_fraction", 0.01),
        growth_penalty_K=raw.get("growth_penalty_K", 0.5),
        growth_penalty_n=raw.get("growth_penalty_n", 2.0),
        name=raw.get("name", "unnamed-project"),
        description=raw.get("description", ""),
        sources=raw.get("sources", []),
    )

Writing rl_strain_design/config.py


In [4]:
%%writefile rl_strain_design/filters.py
"""
filters.py — declarative pre-FBA validity rules for candidate combos
=======================================================================
Any biological knowledge about "which knockout combos are nonsensical or
known-lethal before you even bother running FBA on them" lives here, as
DATA (a list of rule dicts from a config file), not as Python if-statements
buried in the search code. This is what makes the search engine reusable
across projects/organisms: a new team edits their config file, not this
module.

Three rule types cover everything the original iGEM-2026 yeast project
needed (see configs/yeast9_methyl_myristate.yaml for the real example):

  mutual_exclusion_at_order
    "If ALL of these genes are knocked out together, and the combo has
     reached at least this size, it's invalid."
    Example: FAA1 + FAA4 + a third gene is lethal (iGEM project finding).

  conditional_exclusion
    "Exclude this gene whenever something about the TARGET profile makes
     knocking it out self-contradictory."
    Example: don't knock out the only desaturase (OLE1) if the target
    profile wants any of the unsaturated product it makes.

  gated_subset
    "These genes are off-limits unless a named gate is explicitly open."
    Example: iGEM team decision — don't let degradation-pathway genes
    (POX1/PXA1/PXA2) appear in first-iteration wet-lab proposals; only
    allow them once a human flips allow_degradation_knockouts=True for a
    later search iteration.

No eval() or string-executed conditions anywhere — conditions are small
structured comparisons so a config file can't become an arbitrary code
execution surface.
"""

from dataclasses import dataclass
from typing import Any


@dataclass
class Rule:
    kind: str
    params: dict


_OPERATORS = {
    ">": lambda a, b: a > b,
    ">=": lambda a, b: a >= b,
    "<": lambda a, b: a < b,
    "<=": lambda a, b: a <= b,
    "==": lambda a, b: a == b,
    "!=": lambda a, b: a != b,
}


def load_rules(raw_rules: list[dict]) -> list[Rule]:
    """Parse the `pre_fba_filters` section of a config file into Rule objects."""
    rules = []
    for r in raw_rules:
        kind = r.get("type")
        if kind not in ("mutual_exclusion_at_order", "conditional_exclusion", "gated_subset"):
            raise ValueError(
                f"Unknown filter rule type '{kind}'. Must be one of: "
                "mutual_exclusion_at_order, conditional_exclusion, gated_subset"
            )
        rules.append(Rule(kind=kind, params=r))
    return rules


def _check_mutual_exclusion_at_order(combo: frozenset, params: dict) -> bool:
    """True = combo is VALID (not excluded)."""
    genes = set(params["genes"])
    min_order = params["min_order"]
    if genes.issubset(combo) and len(combo) >= min_order:
        return False
    return True


def _check_conditional_exclusion(combo: frozenset, target_profile: dict, params: dict) -> bool:
    """True = combo is VALID (not excluded)."""
    gene = params["gene"]
    if gene not in combo:
        return True
    cond = params["condition"]
    key = cond["target_profile_key"]
    op = cond["operator"]
    value = cond["value"]
    actual = target_profile.get(key, 0)
    if op not in _OPERATORS:
        raise ValueError(f"Unknown operator '{op}' in conditional_exclusion rule")
    condition_true = _OPERATORS[op](actual, value)
    # If the condition holds, this gene's knockout is excluded.
    return not condition_true


def _check_gated_subset(combo: frozenset, params: dict, open_gates: set) -> bool:
    """True = combo is VALID (not excluded)."""
    genes = set(params["genes"])
    gate_name = params["gate_name"]
    default_open = params.get("default", False)
    is_open = gate_name in open_gates if open_gates else default_open
    if not is_open and genes & combo:
        return False
    return True


def is_valid_combo(combo: frozenset, target_profile: dict, rules: list[Rule],
                    open_gates: set | None = None) -> bool:
    """
    Run every rule against a candidate combo. Returns False (invalid) on the
    first rule that rejects it, matching the original project's short-circuit
    behaviour. `open_gates` is the set of gate names a caller has explicitly
    opened (e.g. {"allow_degradation_knockouts"} for a later search
    iteration) -- omit it to use each gate's own config-specified default.
    """
    open_gates = open_gates or set()
    for rule in rules:
        if rule.kind == "mutual_exclusion_at_order":
            if not _check_mutual_exclusion_at_order(combo, rule.params):
                return False
        elif rule.kind == "conditional_exclusion":
            if not _check_conditional_exclusion(combo, target_profile, rule.params):
                return False
        elif rule.kind == "gated_subset":
            if not _check_gated_subset(combo, rule.params, open_gates):
                return False
    return True

Writing rl_strain_design/filters.py


In [5]:
%%writefile rl_strain_design/agent.py
"""
agent.py — generic linear contextual-bandit / Q-learning agent
==================================================================
Not deep RL on purpose: at the scale this is meant for (a handful of
candidate genes, knockout order <=3-4), a linear Q-function is enough to
generalize across target profiles, is fast to train, and -- importantly
for a wet-lab-facing tool -- is inspectable (every weight has a meaning).

Everything project-specific (which genes, which products, how many genes
per combo) is passed in via a SearchConfig; nothing here hardcodes a gene
list or a target-profile schema.
"""

from dataclasses import dataclass, field
import random
import numpy as np

from .config import SearchConfig
from .filters import is_valid_combo


def feature_dim(config: SearchConfig) -> int:
    n_products = len(config.exchange_reactions)
    n_genes = len(config.candidate_genes)
    # [target vec][action one-hot][combo one-hot][bias][progress]
    return n_products + n_genes * 2 + 2


def build_features(config: SearchConfig, target_profile: dict,
                    current_combo: frozenset, action_gene: str) -> np.ndarray:
    """
    Feature vector for the linear Q-function. Layout (sizes depend on the
    config passed in):
      [0 : P]         target profile weights, in the order config.exchange_reactions
                       was declared
      [P : P+G]       one-hot: which gene is being considered as the action
      [P+G : P+2G]    one-hot: which genes are already in current_combo
      [P+2G]          bias term
      [P+2G+1]        len(current_combo) / max_knockout_order  (episode progress)
    """
    products = list(config.exchange_reactions.keys())
    genes = config.candidate_genes
    target_vec = np.array([target_profile.get(p, 0.0) for p in products])
    action_onehot = np.array([1.0 if g == action_gene else 0.0 for g in genes])
    combo_onehot = np.array([1.0 if g in current_combo else 0.0 for g in genes])
    bias = np.array([1.0])
    progress = np.array([len(current_combo) / config.max_knockout_order])
    return np.concatenate([target_vec, action_onehot, combo_onehot, bias, progress])


@dataclass
class LinearQAgent:
    """
    Q(s, a) = w . phi(s, a), trained with 1-step TD updates (SARSA-max,
    i.e. Q-learning). `config` supplies the gene list, max combo order,
    and feature dimension; `open_gates` controls which gated_subset filter
    rules (see filters.py) are relaxed for this agent instance -- e.g. a
    "first iteration" agent leaves a gate closed, a "later iteration" agent
    can be constructed with it open.
    """
    config: SearchConfig
    lr: float = 0.05
    gamma: float = 0.9
    epsilon: float = 0.2
    weights: np.ndarray = None
    open_gates: frozenset = field(default_factory=frozenset)

    def __post_init__(self):
        if self.weights is None:
            self.weights = np.zeros(feature_dim(self.config))

    def q_value(self, target_profile, current_combo, action_gene):
        phi = build_features(self.config, target_profile, current_combo, action_gene)
        return float(self.weights @ phi), phi

    def available_actions(self, target_profile, current_combo):
        remaining = [g for g in self.config.candidate_genes if g not in current_combo]
        valid = [
            g for g in remaining
            if is_valid_combo(frozenset(current_combo | {g}), target_profile,
                               self.config.filter_rules, self.open_gates)
        ]
        return valid

    def select_action(self, target_profile, current_combo, greedy=False):
        actions = self.available_actions(target_profile, current_combo)
        if not actions:
            return None
        if (not greedy) and random.random() < self.epsilon:
            return random.choice(actions)
        q_vals = [self.q_value(target_profile, current_combo, a)[0] for a in actions]
        return actions[int(np.argmax(q_vals))]

    def update(self, target_profile, current_combo, action_gene, reward, next_combo, done):
        q_sa, phi = self.q_value(target_profile, current_combo, action_gene)
        if done:
            target = reward
        else:
            next_actions = self.available_actions(target_profile, next_combo)
            if not next_actions:
                target = reward
            else:
                next_q = max(self.q_value(target_profile, next_combo, a)[0] for a in next_actions)
                target = reward + self.gamma * next_q
        td_error = target - q_sa
        self.weights += self.lr * td_error * phi
        return td_error

Writing rl_strain_design/agent.py


In [6]:
%%writefile rl_strain_design/search.py
"""
search.py — training loop, episode runner, and brute-force validator
========================================================================
Generic over any evaluator object exposing:
    evaluate(knockout_combo: tuple[str, ...], target_profile: dict) -> float
(CobraEvaluator in cobra_evaluator.py is the real one; any object with that
method works, including a trivial mock for testing the RL mechanics alone.)
"""

import itertools
import random
import numpy as np

from .config import SearchConfig
from .agent import LinearQAgent
from .filters import is_valid_combo


def sample_random_target_profile(config: SearchConfig) -> dict:
    """Random plausible target profile over the config's tracked products,
    for training diversity. Drawn from a Dirichlet so entries sum to 1."""
    products = list(config.exchange_reactions.keys())
    weights = np.random.dirichlet(np.ones(len(products)))
    return {p: round(float(w), 3) for p, w in zip(products, weights)}


def run_episode(agent: LinearQAgent, evaluator, target_profile: dict, train: bool = True):
    """
    One episode: greedily/eps-greedily build up a knockout combo one gene at
    a time, up to config.max_knockout_order genes, rewarding the agent for
    each incremental improvement in evaluator score (reward shaping -- the
    episode isn't really a long-horizon MDP, it's "sequentially building a
    set," so shaping on the delta keeps the signal dense).
    """
    config = agent.config
    combo = frozenset()
    prev_score = evaluator.evaluate(tuple(sorted(combo)), target_profile)

    trace = []
    for step in range(config.max_knockout_order):
        action = agent.select_action(target_profile, combo, greedy=not train)
        if action is None:
            break
        next_combo = frozenset(combo | {action})
        new_score = evaluator.evaluate(tuple(sorted(next_combo)), target_profile)
        reward = new_score - prev_score
        done = (step == config.max_knockout_order - 1)

        if train:
            agent.update(target_profile, combo, action, reward, next_combo, done)

        trace.append((action, new_score))
        combo, prev_score = next_combo, new_score

    return combo, prev_score, trace


def train_agent(config: SearchConfig, evaluator, n_episodes: int = 2000, seed: int = 0,
                 open_gates=frozenset(), epsilon_decay: float = 0.999, epsilon_min: float = 0.02):
    random.seed(seed)
    np.random.seed(seed)
    agent = LinearQAgent(config=config, open_gates=open_gates)
    history = []
    for _ in range(n_episodes):
        target_profile = sample_random_target_profile(config)
        _, score, _ = run_episode(agent, evaluator, target_profile, train=True)
        history.append(score)
        agent.epsilon = max(epsilon_min, agent.epsilon * epsilon_decay)
    return agent, history


def brute_force_best(config: SearchConfig, evaluator, target_profile: dict,
                      max_order: int = None, open_gates=frozenset()):
    """Exhaustively try every combo up to max_order (defaults to
    config.max_knockout_order), applying the same filter rules the agent
    uses, and return the best-scoring one. The ground-truth baseline the RL
    agent is checked against."""
    max_order = max_order or config.max_knockout_order
    best_combo, best_score = None, -1.0
    for order in range(1, max_order + 1):
        for combo in itertools.combinations(config.candidate_genes, order):
            if not is_valid_combo(frozenset(combo), target_profile,
                                   config.filter_rules, open_gates):
                continue
            score = evaluator.evaluate(combo, target_profile)
            if score > best_score:
                best_combo, best_score = combo, score
    return best_combo, best_score


def evaluate_agent_vs_bruteforce(agent: LinearQAgent, evaluator, n_test_profiles: int = 25,
                                  seed: int = 1, open_gates=frozenset(), verbose: bool = True):
    """Sanity-checks the trained agent against the brute-force optimum on
    fresh random target profiles. Returns the list of (target, rl_combo,
    rl_score, bf_combo, bf_score, gap) tuples; gap = bf_score - rl_score,
    0 meaning the agent matched brute force exactly."""
    config = agent.config
    random.seed(seed)
    np.random.seed(seed)
    rows = []
    for _ in range(n_test_profiles):
        target_profile = sample_random_target_profile(config)
        rl_combo, rl_score, _ = run_episode(agent, evaluator, target_profile, train=False)
        bf_combo, bf_score = brute_force_best(config, evaluator, target_profile,
                                               open_gates=open_gates)
        gap = bf_score - rl_score
        rows.append((target_profile, rl_combo, rl_score, bf_combo, bf_score, gap))

    if verbose:
        print(f"\n{'Target Profile':<45} {'RL combo':<25} {'RL score':<10} "
              f"{'Brute-force combo':<25} {'BF score':<10} {'Gap'}")
        print("-" * 130)
        for target_profile, rl_combo, rl_score, bf_combo, bf_score, gap in rows:
            tp_str = ", ".join(f"{k}:{v:.2f}" for k, v in target_profile.items() if v > 0.01)
            print(f"{tp_str:<45} {str(sorted(rl_combo)):<25} {rl_score:<10.3f} "
                  f"{str(sorted(bf_combo)):<25} {bf_score:<10.3f} {gap:.3f}")
        print("-" * 130)
        gaps = [r[-1] for r in rows]
        print(f"Mean gap (BF score - RL score): {np.mean(gaps):.4f}  "
              f"(closer to 0 = RL agent nearly matches brute force)")

    return rows


def greedy_search(config: SearchConfig, evaluator, target_profile: dict,
                   max_order: int = None, open_gates=frozenset()):
    """Add the single best next gene at each step, stopping when none helps.

    RECOMMENDED DEFAULT. The benchmark in benchmark.py found that on
    synthetic landscapes with main effects, sparse pairwise epistasis and a
    growth penalty, this strategy reached 99.4-99.9% of the exhaustive
    optimum across five epistasis levels and five seeds, using roughly
    2 x n_genes evaluations -- while the linear Q-learning agent in agent.py
    reached only 80-85% with much higher variance and far more evaluations.

    Use the RL agent only if you can show on your own landscape that it beats
    this. On present evidence it does not.
    """
    max_order = max_order or config.max_knockout_order
    combo = ()
    best = float(evaluator.evaluate(combo, target_profile))
    for _ in range(max_order):
        cand, cand_score = None, best
        for g in config.candidate_genes:
            if g in combo:
                continue
            trial = tuple(sorted(combo + (g,)))
            if not is_valid_combo(frozenset(trial), target_profile,
                                   config.filter_rules, open_gates):
                continue
            s = float(evaluator.evaluate(trial, target_profile))
            if s > cand_score:
                cand, cand_score = trial, s
        if cand is None:
            break
        combo, best = cand, cand_score
    return combo, best

Writing rl_strain_design/search.py


In [7]:
%%writefile rl_strain_design/cobra_evaluator.py
"""
cobra_evaluator.py — generic COBRApy-based fitness evaluator
=================================================================
Scores a candidate knockout combo against a target product-ratio profile,
for ANY COBRApy-compatible genome-scale model -- not hardcoded to any one
organism. Everything organism-specific (which model file, which genes,
which reactions, which growth-penalty shape) comes from a SearchConfig
(see config.py), not from constants in this file.

THE OBJECTIVE FORMULATION (why it's 3 stages, not 1):

A naive "maximize a weighted sum of target reactions" objective always
degenerates to an all-or-nothing corner solution in linear programming --
confirmed empirically on the original iGEM yeast project: the same
collapse happened at every weight ratio tested. Fixed with a lexicographic
3-stage solve instead:

  Stage 1: maximize total achievable flux across all target reactions -> T_max
  Stage 2: require >= alpha * T_max total; minimize L1 deviation from the
           target RATIO (not the raw weighted sum) -- this is what actually
           produces a blend matching the target composition
  Stage 3: lock that ratio-match in place (deviation <= its achieved
           minimum), then maximize growth without sacrificing it

Stage 3 exists because an earlier version of this evaluator just removed
the growth constraint outright and let the LP decide -- it had zero
incentive to keep any growth above the floor, so every combo collapsed to
minimum viable growth and the growth penalty below became a near-constant
multiplier with no power to differentiate between combos. Locking the
product-ratio match first, then asking "how much growth can we still
afford," gives a genuinely combo-dependent growth value.

THE GROWTH PENALTY:

Rather than a hard pass/fail cutoff (which gives zero gradient signal to
an RL agent near the viability boundary), growth loss is penalized
smoothly with a Hill-type function -- the same functional shape used for
toxicity terms in dynamic metabolic-control literature (e.g. Ohkubo et
al. 2024's T_X term), adapted here for a steady-state FBA context:

    g = achieved_growth / wt_growth              (clipped to [0, 1])
    deficit d = 1 - g
    penalty(d) = d^n / (K^n + d^n)                Hill function
    final_score = cosine_similarity(predicted, target) * (1 - penalty)

A tiny hard floor (config: hard_min_growth_fraction) still rejects
genuinely dead strains outright -- a smooth curve shouldn't paper over an
actually-infeasible genotype.
"""

import warnings
warnings.filterwarnings("ignore")
import numpy as np
from cobra.io import read_sbml_model

from .config import SearchConfig
from .filters import is_valid_combo


def cosine_similarity(predicted: dict, target: dict) -> float:
    fas = list(target.keys())
    p = np.array([predicted.get(k, 0.0) for k in fas])
    t = np.array([target.get(k, 0.0) for k in fas])
    if np.linalg.norm(p) < 1e-12 or np.linalg.norm(t) < 1e-12:
        return 0.0
    return float(np.dot(p, t) / (np.linalg.norm(p) * np.linalg.norm(t)))


def growth_penalty(achieved_growth: float, wt_growth: float, K: float, n: float) -> float:
    """Smooth Hill-type multiplier in [0, 1]. 1.0 = no growth lost."""
    if wt_growth <= 0:
        return 0.0
    g = max(0.0, min(1.0, achieved_growth / wt_growth))
    d = 1.0 - g
    penalty = (d ** n) / (K ** n + d ** n)
    return 1.0 - penalty


class CobraEvaluator:
    """
    Loads the model once (expensive) and exposes an `evaluate()` method
    matching the interface the search engine expects:
        evaluate(knockout_combo: tuple[str, ...], target_profile: dict) -> float
    Results are cached by (combo, target_profile) since FBA solves are the
    expensive step and the same combo/target pair is often re-queried
    during RL training and validation.
    """

    def __init__(self, config: SearchConfig):
        self.config = config
        print(f"Loading model for '{config.name}' ({config.model_path})...")
        self._model = read_sbml_model(config.model_path)
        wt_sol = self._model.optimize()
        self._wt_growth = wt_sol.objective_value
        print(f"Loaded. WT growth rate: {self._wt_growth:.6f}")
        self._cache: dict = {}
        self._parts: dict = {}   # key -> (raw cosine, achieved growth)

    def is_valid(self, combo: frozenset, target_profile: dict, open_gates: set | None = None) -> bool:
        return is_valid_combo(combo, target_profile, self.config.filter_rules, open_gates)

    def diagnose(self, knockout_combo, target_profile: dict) -> dict:
        """Break a score into its parts: raw cosine match, growth as a fraction
        of wild-type, growth multiplier, and final score. For debugging/reporting."""
        score = self.evaluate(knockout_combo, target_profile)
        key = (tuple(sorted(knockout_combo)), tuple(sorted(target_profile.items())))
        raw, g = self._parts.get(key, (0.0, 0.0))
        return {"cosine": raw, "growth_fraction": g / self._wt_growth if self._wt_growth else 0.0,
                "score": score}

    def evaluate(self, knockout_combo, target_profile: dict) -> float:
        key = (tuple(sorted(knockout_combo)), tuple(sorted(target_profile.items())))
        if key in self._cache:
            return self._cache[key]

        cfg = self.config
        with self._model as model:
            for gene_common in knockout_combo:
                model_id = cfg.gene_to_model_id.get(gene_common)
                if model_id is None:
                    continue
                try:
                    model.genes.get_by_id(model_id).knock_out()
                except KeyError:
                    continue

            biomass_rxn = model.reactions.get_by_id(cfg.biomass_reaction_id)
            biomass_rxn.lower_bound = cfg.hard_min_growth_fraction * self._wt_growth

            target_rxns = {
                name: model.reactions.get_by_id(rid)
                for name, rid in cfg.exchange_reactions.items() if name in target_profile
            }
            if not target_rxns:
                self._cache[key] = 0.0
                return 0.0

            # Stage 1: max achievable total output across target reactions
            model.objective = {rxn: 1.0 for rxn in target_rxns.values()}
            model.objective.direction = "max"
            T_max = model.slim_optimize()
            if T_max is None or T_max < 1e-9:
                self._cache[key] = 0.0
                return 0.0

            # Stage 2: floor total output, minimize L1 deviation from target ratio
            total_con = model.problem.Constraint(
                sum(rxn.flux_expression for rxn in target_rxns.values()),
                lb=cfg.ratio_match_alpha * T_max, name="min_total_output")
            model.add_cons_vars([total_con])

            dev_vars = [model.problem.Variable(f"dev_{name}", lb=0) for name in target_rxns]
            model.add_cons_vars(dev_vars)
            for name, rxn in target_rxns.items():
                d = model.variables[f"dev_{name}"]
                target_flux = target_profile[name] * T_max
                model.add_cons_vars([model.problem.Constraint(rxn.flux_expression - d, ub=target_flux)])
                model.add_cons_vars([model.problem.Constraint(rxn.flux_expression + d, lb=target_flux)])

            model.objective = sum(model.variables[f"dev_{name}"] for name in target_rxns)
            model.objective.direction = "min"
            min_dev = model.slim_optimize()
            if min_dev is None:
                self._cache[key] = 0.0
                return 0.0

            # Stage 3: lock the ratio match, then maximize growth
            dev_lock = model.problem.Constraint(
                sum(model.variables[f"dev_{name}"] for name in target_rxns),
                ub=min_dev, name="lock_ratio_match")
            model.add_cons_vars([dev_lock])
            model.objective = biomass_rxn
            model.objective.direction = "max"
            sol = model.optimize()

            if sol.status != "optimal":
                self._cache[key] = 0.0
                return 0.0

            predicted_profile = {
                name: max(0.0, sol.fluxes.get(rid, 0.0))
                for name, rid in cfg.exchange_reactions.items()
            }
            achieved_growth = sol.fluxes.get(cfg.biomass_reaction_id, 0.0)

        raw_score = cosine_similarity(predicted_profile, target_profile)
        penalty_mult = growth_penalty(achieved_growth, self._wt_growth,
                                       cfg.growth_penalty_K, cfg.growth_penalty_n)
        score = raw_score * penalty_mult
        self._parts[key] = (raw_score, achieved_growth)
        self._cache[key] = score
        return score

Writing rl_strain_design/cobra_evaluator.py


In [8]:
%%writefile rl_strain_design/diagnostics.py
"""
diagnostics.py — pre-flight checks on a strain-design SCORING SETUP
=====================================================================
WHAT THIS IS FOR, AND WHAT IT IS NOT

Model-quality test suites already exist and are excellent: memote (Lieven et
al. 2020, doi:10.1038/s41587-020-0446-y) and FBCModelTests.jl check whether
your genome-scale model is well-formed -- annotations, mass balance,
stoichiometric consistency, energy-generating cycles, unbounded fluxes. FROG
reports (Olivier/Bergmann et al. 2024) check that two software stacks
reproduce the same numbers from the same model.

None of them check the thing that silently ruined our own first pipeline:
the OBJECTIVE AND SCORING SETUP the user wrote on top of a perfectly good
model. A scoring function can be syntactically valid, run without error,
produce confident-looking rankings, and still be measuring nothing. That is
the gap this module targets.

HONEST NOVELTY ACCOUNTING (read before citing this as novel anywhere)

Of the six checks below, only two are not automated by any tool we could
find, and one design behaviour is unclaimed:

  NEW       D1 null_intervention_parity  -- StrainDesign's docs warn in prose
                                            that "the MILP may return the wild
                                            type", but no tool automates the
                                            check.
  NEW       D2 landscape_discrimination  -- no tool we found reports whether
                                            its own score can separate
                                            candidates at all.
  NEW       output gating (see proposals.py) -- no published strain-design tool
                                            refuses to emit designs when its
                                            own scoring landscape is
                                            uninformative.

  KNOWN     D3 scalarization_degeneracy  -- this is Flux Variability Analysis
                                            (Mahadevan & Schilling 2003) and
                                            alternate-optima analysis (Reed &
                                            Palsson 2004) applied to a
                                            multi-product objective. The
                                            underlying result that weighted
                                            sums only reach vertices is Das &
                                            Dennis 1997 (doi:10.1007/BF01197559).
                                            We automate and explain it; we did
                                            not discover it.
  KNOWN     D4 free_production           -- memote already ships
                                            find_reactions_with_unbounded_flux_
                                            default_condition and
                                            detect_energy_generating_cycles.
                                            Run memote too. Ours is a
                                            convenience check on the specific
                                            target reactions, nothing more.
                                            See Fritzemeier et al. 2017,
                                            Thiele & Palsson 2010.
  KNOWN     D5 growth_confound           -- objective choice driving
                                            predictions is Schuetz et al. 2007
                                            and Feist & Palsson 2010.
                                            IMPORTANT: growth-coupled
                                            production is the INTENDED
                                            mechanism of most strain designs
                                            (Klamt & Mahadevan 2015;
                                            von Kamp & Klamt 2017), so this
                                            check can never return FAIL --
                                            only a WARN asking whether the
                                            coupling is what you meant.
  KNOWN     D6 parameter_sensitivity     -- ordinary sensitivity analysis.

Do not describe this suite as a whole as novel. The defensible claim is:
"a pre-flight suite for the scoring setup rather than the model, which gates
its own output; two of its checks and the gating behaviour appear to be
unautomated elsewhere."
"""

from dataclasses import dataclass, field
import itertools
from typing import Any

import numpy as np

# Below this, two scores are indistinguishable from solver noise rather than
# genuinely different. LP solvers typically converge to ~1e-9 but FBA scores
# built from several solves accumulate more error than that.
SOLVER_NOISE_FLOOR = 1e-6

# THE THRESHOLD THAT ACTUALLY MATTERS, AND WHY IT IS NOT THE SOLVER'S
#
# Our first version of this module compared score differences against
# SOLVER_NOISE_FLOOR, and it let our own broken pipeline through with a
# PASS-equivalent: the best knockout beat doing nothing by 2e-5, which is
# comfortably above 1e-6, so the check said "fine" about a landscape we
# already knew was flat.
#
# Solver tolerance is the wrong standard. The question a strain-design score
# has to answer is not "is this difference real arithmetic?" but "could the
# wet lab ever tell these two strains apart?" GC-MS fatty-acid composition,
# reported as a fraction of total peak area, carries roughly 1-5% relative
# error between biological replicates. A model that ranks two designs 0.01%
# apart is making a distinction no experiment can test, so for the purpose of
# proposing bench work the landscape is flat even though the arithmetic is
# sound.
#
# These are therefore RELATIVE thresholds tied to measurement resolution, and
# they are deliberately conservative. Override them for your own assay:
# a method with tighter replicate error earns a smaller threshold.
MEASURABLE_RELATIVE_MARGIN = 0.05   # 5% -- a difference worth building a strain for
DETECTABLE_RELATIVE_MARGIN = 0.01   # 1% -- the optimistic edge of GC-MS resolution

PASS, WARN, FAIL, SKIP = "PASS", "WARN", "FAIL", "SKIP"


@dataclass
class CheckResult:
    check_id: str
    name: str
    status: str
    headline: str           # one plain-language sentence
    detail: str = ""        # what was measured
    fix: str = ""           # what to do about it
    evidence: dict = field(default_factory=dict)
    citations: list = field(default_factory=list)
    novelty: str = "known"  # "new" or "known" -- see module docstring

    @property
    def blocking(self) -> bool:
        """A FAIL on a blocking check means downstream rankings are unusable."""
        return self.status == FAIL and self.check_id in ("D1", "D2")


@dataclass
class DiagnosticReport:
    results: list
    target_profile: dict
    n_candidates_tested: int

    @property
    def blocking_failures(self) -> list:
        return [r for r in self.results if r.blocking]

    @property
    def usable(self) -> bool:
        """True when the scoring setup can support a ranking of candidates."""
        return not self.blocking_failures

    def counts(self) -> dict:
        out = {PASS: 0, WARN: 0, FAIL: 0, SKIP: 0}
        for r in self.results:
            out[r.status] += 1
        return out

    def to_text(self) -> str:
        lines = []
        w = 74
        lines.append("=" * w)
        lines.append("SCORING-SETUP DIAGNOSTIC REPORT")
        lines.append("=" * w)
        tp = ", ".join(f"{k}={v}" for k, v in self.target_profile.items())
        lines.append(f"Target profile    : {tp}")
        lines.append(f"Candidates tested : {self.n_candidates_tested}")
        c = self.counts()
        lines.append(f"Result            : {c[PASS]} pass, {c[WARN]} warn, "
                     f"{c[FAIL]} fail, {c[SKIP]} skipped")
        lines.append("")
        verdict = ("USABLE -- rankings from this scoring setup are meaningful"
                   if self.usable else
                   "NOT USABLE -- a blocking check failed; rankings are "
                   "meaningless")
        lines.append(f"VERDICT: {verdict}")
        lines.append("=" * w)
        for r in self.results:
            tag = "[new]" if r.novelty == "new" else ""
            lines.append("")
            lines.append(f"{r.status:5} {r.check_id}  {r.name} {tag}".rstrip())
            lines.append(f"      {r.headline}")
            if r.detail:
                for ln in r.detail.split("\n"):
                    lines.append(f"      {ln}")
            if r.status in (WARN, FAIL) and r.fix:
                lines.append(f"      FIX: {r.fix}")
            if r.citations:
                lines.append(f"      refs: {'; '.join(r.citations)}")
        lines.append("")
        lines.append("=" * w)
        lines.append("Run memote (doi:10.1038/s41587-020-0446-y) as well -- it")
        lines.append("checks the MODEL; this suite checks the SCORING SETUP.")
        lines.append("=" * w)
        return "\n".join(lines)

    def to_markdown(self) -> str:
        icon = {PASS: "PASS", WARN: "WARN", FAIL: "FAIL", SKIP: "n/a"}
        lines = ["# Scoring-setup diagnostic report", ""]
        verdict = ("**Usable.** Rankings from this scoring setup are meaningful."
                   if self.usable else
                   "**Not usable.** A blocking check failed, so any ranking "
                   "this setup produces is meaningless.")
        lines += [verdict, "",
                  f"Target profile: `{self.target_profile}`  ",
                  f"Candidates tested: {self.n_candidates_tested}", "",
                  "| Check | Result | Finding |", "| --- | --- | --- |"]
        for r in self.results:
            head = r.headline.replace("|", "/")
            lines.append(f"| {r.check_id} {r.name} | {icon[r.status]} | {head} |")
        lines += ["", "## Detail", ""]
        for r in self.results:
            lines.append(f"### {r.check_id} {r.name} — {icon[r.status]}")
            lines.append(r.headline)
            if r.detail:
                lines += ["", "```", r.detail, "```"]
            if r.status in (WARN, FAIL) and r.fix:
                lines += ["", f"**Fix:** {r.fix}"]
            if r.citations:
                lines += ["", f"*References: {'; '.join(r.citations)}*"]
            lines.append("")
        return "\n".join(lines)


# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------

def _default_candidates(evaluator, target_profile, max_order=2, cap=40):
    """Candidate combos to probe, taken from the evaluator's own config and
    filters so we test exactly what the search would actually consider."""
    cfg = getattr(evaluator, "config", None)
    if cfg is None:
        raise ValueError(
            "evaluator has no .config; pass candidate_combos= explicitly")
    genes = cfg.candidate_genes
    order_cap = min(max_order, cfg.max_knockout_order)
    out = []
    for order in range(1, order_cap + 1):
        for combo in itertools.combinations(genes, order):
            if hasattr(evaluator, "is_valid") and not evaluator.is_valid(
                    frozenset(combo), target_profile):
                continue
            out.append(combo)
    if len(out) > cap:
        step = len(out) / cap
        out = [out[int(i * step)] for i in range(cap)]
    return out


def _score_all(evaluator, combos, target_profile):
    return {c: float(evaluator.evaluate(c, target_profile)) for c in combos}


# ---------------------------------------------------------------------------
# D1 -- null-intervention parity                                       [NEW]
# ---------------------------------------------------------------------------

def check_null_intervention_parity(evaluator, target_profile, scores) -> CheckResult:
    """Does doing nothing score as well as the best intervention?

    If the unedited strain scores as highly as the best knockout, the score
    is not measuring the effect of the intervention. Every ranking built on
    it is then an artefact. This is the single most informative check in the
    suite and the cheapest to run.
    """
    null_score = float(evaluator.evaluate((), target_profile))
    if not scores:
        return CheckResult("D1", "null-intervention parity", SKIP,
                           "No candidates to compare against.", novelty="new")
    best_combo = max(scores, key=scores.get)
    best = scores[best_combo]
    margin = best - null_score
    rel = margin / abs(null_score) if abs(null_score) > 1e-12 else float("inf")

    ev = {"null_score": null_score, "best_score": best,
          "best_combo": list(best_combo), "margin": margin,
          "relative_margin": rel}
    detail = (f"no-intervention score : {null_score:.6f}\n"
              f"best candidate        : {list(best_combo)} -> {best:.6f}\n"
              f"margin                : {margin:.3e}  "
              f"({rel*100:.3f}% of baseline)")
    cites = ["StrainDesign docs note the wild type can be returned as a "
             "solution (doi:10.1093/bioinformatics/btac632)"]

    fix = ("The objective is essentially satisfiable without any edit. "
           "Constrain the route the product must take (e.g. require flux "
           "through the synthesis pathway), bound the product exchange "
           "realistically, or score product yield at fixed growth instead of "
           "profile shape.")

    if margin <= SOLVER_NOISE_FLOOR:
        return CheckResult(
            "D1", "null-intervention parity", FAIL,
            "Doing nothing scores as well as the best intervention, so the "
            "score is not measuring the intervention at all.",
            detail, fix, ev, cites, novelty="new")
    if rel < DETECTABLE_RELATIVE_MARGIN:
        return CheckResult(
            "D1", "null-intervention parity", FAIL,
            f"The best intervention beats doing nothing by only "
            f"{rel*100:.3f}%, which is below what the assay could resolve, so "
            "the ranking is untestable even though the arithmetic is sound.",
            detail, fix, ev, cites, novelty="new")
    if rel < MEASURABLE_RELATIVE_MARGIN:
        return CheckResult(
            "D1", "null-intervention parity", WARN,
            f"The best intervention beats doing nothing by {rel*100:.2f}%, "
            "near the edge of replicate measurement error.",
            detail, "Treat the ranking as provisional and check the margin "
            "against your assay's own replicate error before committing bench "
            "time.", ev, cites, novelty="new")
    return CheckResult(
        "D1", "null-intervention parity", PASS,
        f"The best intervention beats doing nothing by {rel*100:.1f}%, a "
        "margin an experiment could resolve.",
        detail, evidence=ev, citations=cites, novelty="new")


# ---------------------------------------------------------------------------
# D2 -- landscape discrimination                                       [NEW]
# ---------------------------------------------------------------------------

def check_landscape_discrimination(evaluator, target_profile, scores) -> CheckResult:
    """Can the score tell candidates apart at all?

    Reports the spread of scores across candidates and how many genuinely
    distinguishable tiers exist once solver noise is accounted for. A search
    over a landscape with one tier is a random number generator with extra
    steps.
    """
    if len(scores) < 2:
        return CheckResult("D2", "landscape discrimination", SKIP,
                           "Need at least two candidates.", novelty="new")
    vals = np.array(sorted(scores.values()))
    spread = float(vals[-1] - vals[0])
    std = float(np.std(vals))
    scale = float(np.median(np.abs(vals)))
    rel_spread = spread / scale if scale > 1e-12 else float("inf")

    # Two tier counts, because they answer different questions: what the
    # arithmetic can separate, and what an experiment could separate.
    def count_tiers(min_gap):
        tiers, last = 1, vals[0]
        for v in vals[1:]:
            if v - last > min_gap:
                tiers += 1
                last = v
        return tiers

    tiers_numeric = count_tiers(SOLVER_NOISE_FLOOR)
    tiers_measurable = count_tiers(DETECTABLE_RELATIVE_MARGIN * scale)

    ev = {"n": len(vals), "min": float(vals[0]), "max": float(vals[-1]),
          "spread": spread, "relative_spread": rel_spread, "std": std,
          "tiers_numeric": tiers_numeric,
          "tiers_measurable": tiers_measurable}
    detail = (f"candidates              : {len(vals)}\n"
              f"score range             : {vals[0]:.6f} .. {vals[-1]:.6f}\n"
              f"spread                  : {spread:.3e} "
              f"({rel_spread*100:.3f}% of typical score)\n"
              f"std deviation           : {std:.3e}\n"
              f"tiers the solver can see: {tiers_numeric} of {len(vals)}\n"
              f"tiers an assay can see  : {tiers_measurable} of {len(vals)}")
    cites = ["Bernstein et al. 2021 (doi:10.1186/s13059-021-02289-z)",
             "Kaste & Shachar-Hill 2024 (arXiv:2303.12651)"]

    if spread <= SOLVER_NOISE_FLOOR:
        return CheckResult(
            "D2", "landscape discrimination", FAIL,
            "Every candidate scores identically, so the score cannot rank "
            "anything.",
            detail, "The score is flat. Find what it is actually responding "
            "to (run D5), then redefine it so the intervention changes it.",
            ev, cites, novelty="new")
    if tiers_measurable <= 1:
        return CheckResult(
            "D2", "landscape discrimination", FAIL,
            f"The solver separates these candidates into {tiers_numeric} "
            "levels, but all of them fall inside one measurement-resolution "
            "band, so no experiment could confirm the ordering.",
            detail, "The ranking is numerically real and experimentally "
            "meaningless. Redefine the score so differences exceed assay "
            "error, or report a single undifferentiated set.",
            ev, cites, novelty="new")
    if tiers_measurable <= 2:
        return CheckResult(
            "D2", "landscape discrimination", WARN,
            f"Only {tiers_measurable} measurement-resolvable levels across "
            f"{len(vals)} candidates.",
            detail, "A ranking this coarse can order tiers but not candidates "
            "within a tier. Report tiers, not a ranked list.",
            ev, cites, novelty="new")
    return CheckResult(
        "D2", "landscape discrimination", PASS,
        f"The score separates candidates into {tiers_measurable} levels an "
        "experiment could resolve.",
        detail, evidence=ev, citations=cites, novelty="new")


# ---------------------------------------------------------------------------
# D3 -- scalarization degeneracy                                     [KNOWN]
# ---------------------------------------------------------------------------

def check_scalarization_degeneracy(evaluator, target_profile,
                                    n_weight_samples=5) -> CheckResult:
    """Does a weighted-sum objective over the target products collapse?

    NOTE ON WHAT THIS TESTS: this check builds a weighted-sum objective on
    the model directly and sweeps the weights. It is therefore a test of the
    weighted-sum FORMULATION, not of whatever objective your evaluator
    currently uses. A FAIL means "if you scalarize these products with
    weights, the weights will not control the blend" -- which is worth
    knowing even if you have already moved to goal programming, because it
    documents why you had to.

    This is FVA / alternate-optima analysis applied to a multi-product
    objective: a linear program only ever returns a vertex, so maximizing a
    weighted sum of product exchanges gives you the single best-paying
    product at almost every weight ratio, not a blend. Known result, not our
    discovery -- see the citations.
    """
    model = getattr(evaluator, "_model", None)
    cfg = getattr(evaluator, "config", None)
    cites = ["Das & Dennis 1997 (doi:10.1007/BF01197559)",
             "Mahadevan & Schilling 2003 (doi:10.1016/j.ymben.2003.09.002)",
             "Reed & Palsson 2004 (doi:10.1101/gr.2546004)"]
    if model is None or cfg is None:
        return CheckResult("D3", "scalarization degeneracy", SKIP,
                           "Needs a COBRA model on the evaluator.",
                           citations=cites)

    products = [p for p in cfg.exchange_reactions if p in target_profile]
    if len(products) < 2:
        return CheckResult("D3", "scalarization degeneracy", SKIP,
                           "Only one target product; nothing to scalarize.",
                           citations=cites)

    rng = np.random.default_rng(0)
    supports = []
    rows = []
    try:
        for i in range(n_weight_samples):
            w = (np.ones(len(products)) / len(products) if i == 0
                 else rng.dirichlet(np.ones(len(products))))
            with model as m:
                rxns = {p: m.reactions.get_by_id(cfg.exchange_reactions[p])
                        for p in products}
                m.objective = {rxns[p]: float(w[j])
                               for j, p in enumerate(products)}
                m.objective.direction = "max"
                sol = m.optimize()
                if sol.status != "optimal":
                    continue
                fl = {p: max(0.0, sol.fluxes.get(
                    cfg.exchange_reactions[p], 0.0)) for p in products}
            total = sum(fl.values())
            active = frozenset(p for p, v in fl.items()
                               if total > 0 and v / total > 0.01)
            supports.append(active)
            rows.append((np.round(w, 2).tolist(), len(active),
                         sorted(active)))
    except Exception as exc:  # noqa: BLE001 -- diagnostics must not crash a run
        return CheckResult("D3", "scalarization degeneracy", SKIP,
                           f"Could not run the weight sweep: {exc}",
                           citations=cites)

    if not supports:
        return CheckResult("D3", "scalarization degeneracy", SKIP,
                           "No optimal solutions in the weight sweep.",
                           citations=cites)

    sizes = [len(s) for s in supports]
    detail = "\n".join(
        f"weights {r[0]} -> {r[1]} product(s) active: {r[2]}" for r in rows)
    ev = {"n_samples": len(supports), "support_sizes": sizes,
          "products": products}

    if max(sizes) <= 1:
        return CheckResult(
            "D3", "scalarization degeneracy", FAIL,
            "A weighted sum over these products cannot control the blend: at "
            "every weight tested the solver produced just one product. (This "
            "tests the weighted-sum formulation itself, not your current "
            "objective.)",
            detail, "Do not use a weighted sum for a ratio target. Use "
            "lexicographic / goal programming: maximize total output first, "
            "then minimize deviation from the target ratio. Or inspect the "
            "trade-off surface with COBRApy's production_envelope.",
            ev, cites)
    if len(set(supports)) == 1 and len(supports) > 1:
        return CheckResult(
            "D3", "scalarization degeneracy", WARN,
            "The same set of products stays active at every weight, so the "
            "weights have little influence.",
            detail, "Confirm the weights are doing what you expect; consider "
            "a goal-programming formulation.", ev, cites)
    return CheckResult(
        "D3", "scalarization degeneracy", PASS,
        "Different weights produce different product blends.",
        detail, evidence=ev, citations=cites)


# ---------------------------------------------------------------------------
# D4 -- free production / unbounded export                           [KNOWN]
# ---------------------------------------------------------------------------

def check_free_production(evaluator, target_profile) -> CheckResult:
    """Can the target be produced without consuming anything?

    memote already ships this class of check
    (find_reactions_with_unbounded_flux_default_condition,
    detect_energy_generating_cycles) and you should run memote too. This is a
    convenience check aimed only at the specific target reactions in your
    config, so it appears in the same report as the rest.
    """
    model = getattr(evaluator, "_model", None)
    cfg = getattr(evaluator, "config", None)
    cites = ["memote (doi:10.1038/s41587-020-0446-y) -- run it as well",
             "Fritzemeier et al. 2017 (doi:10.1371/journal.pcbi.1005494)",
             "Thiele & Palsson 2010 (doi:10.1038/nprot.2009.203)"]
    if model is None or cfg is None:
        return CheckResult("D4", "free production", SKIP,
                           "Needs a COBRA model on the evaluator.",
                           citations=cites)

    findings, rows = [], []
    try:
        for name, rid in cfg.exchange_reactions.items():
            if name not in target_profile:
                continue
            rxn = model.reactions.get_by_id(rid)
            ub, lb = rxn.upper_bound, rxn.lower_bound
            huge = abs(ub) >= 1000
            rows.append(f"{name:8} {rid:10} bounds [{lb:g}, {ub:g}]"
                        f"{'   <- effectively unbounded' if huge else ''}")
            if huge:
                findings.append(f"{name} ({rid}) upper bound {ub:g}")
    except Exception as exc:  # noqa: BLE001
        return CheckResult("D4", "free production", SKIP,
                           f"Could not inspect bounds: {exc}", citations=cites)

    detail = "\n".join(rows)
    ev = {"effectively_unbounded": findings}
    if findings:
        return CheckResult(
            "D4", "free production", WARN,
            f"{len(findings)} target exchange reaction(s) are effectively "
            "unbounded, so the model may produce the target more freely than "
            "a cell could.",
            detail, "Set product exchange upper bounds from measured or "
            "literature secretion rates, and run memote's unbounded-flux and "
            "energy-cycle checks on the model.", ev, cites)
    return CheckResult(
        "D4", "free production", PASS,
        "Target exchange reactions carry finite, plausible bounds.",
        detail, evidence=ev, citations=cites)


# ---------------------------------------------------------------------------
# D5 -- growth confound                                              [KNOWN]
# ---------------------------------------------------------------------------

def check_growth_confound(evaluator, target_profile, scores) -> CheckResult:
    """Is the score tracking product formation, or only retained growth?

    CAREFUL: growth-coupled production is the INTENDED mechanism of most
    strain designs (Klamt & Mahadevan 2015; von Kamp & Klamt 2017), so
    correlation between score and growth is not by itself a defect. This
    check therefore never returns FAIL. What it flags is the specific case
    where the product term contributes no variance at all -- the score moves
    only with growth, so the product half of the objective is doing no work.
    """
    cites = ["Schuetz et al. 2007 (doi:10.1038/msb4100162)",
             "Feist & Palsson 2010 (doi:10.1016/j.mib.2010.03.003)",
             "Klamt & Mahadevan 2015 (doi:10.1016/j.ymben.2015.05.006) -- "
             "growth coupling is often intended"]
    if not hasattr(evaluator, "diagnose"):
        return CheckResult("D5", "growth confound", SKIP,
                           "Evaluator has no diagnose() method to decompose "
                           "the score.", citations=cites)
    if len(scores) < 3:
        return CheckResult("D5", "growth confound", SKIP,
                           "Need at least three candidates.", citations=cites)

    prod, growth, total = [], [], []
    try:
        for combo in scores:
            d = evaluator.diagnose(combo, target_profile)
            prod.append(d.get("cosine", 0.0))
            growth.append(d.get("growth_fraction", 0.0))
            total.append(d.get("score", scores[combo]))
    except Exception as exc:  # noqa: BLE001
        return CheckResult("D5", "growth confound", SKIP,
                           f"Could not decompose scores: {exc}",
                           citations=cites)

    prod, growth, total = np.array(prod), np.array(growth), np.array(total)
    pv, gv = float(np.std(prod)), float(np.std(growth))
    # Judge each term's variation against its own scale, so "the product term
    # technically moves by 1e-5" does not count as the term doing work.
    p_scale = max(float(np.median(np.abs(prod))), 1e-12)
    g_scale = max(float(np.median(np.abs(growth))), 1e-12)
    p_inert = (pv / p_scale) < DETECTABLE_RELATIVE_MARGIN
    g_inert = (gv / g_scale) < DETECTABLE_RELATIVE_MARGIN

    def corr(a, b):
        if np.std(a) <= 1e-12 or np.std(b) <= 1e-12:
            return float("nan")
        return float(np.corrcoef(a, b)[0, 1])

    cg, cp = corr(total, growth), corr(total, prod)
    ev = {"product_match_std": pv, "growth_std": gv,
          "corr_score_growth": cg, "corr_score_product": cp}
    detail = (f"product-match (cosine) spread : {pv:.3e} "
              f"({pv/p_scale*100:.3f}% of its scale"
              f"{', inert' if p_inert else ''})\n"
              f"growth-fraction spread        : {gv:.3e} "
              f"({gv/g_scale*100:.3f}% of its scale"
              f"{', inert' if g_inert else ''})\n"
              f"corr(score, growth)           : {cg:.3f}\n"
              f"corr(score, product match)    : {cp:.3f}")
    ev |= {"product_inert": bool(p_inert), "growth_inert": bool(g_inert)}

    if p_inert and not g_inert:
        return CheckResult(
            "D5", "growth confound", WARN,
            "The product-match term is identical for every candidate, so the "
            "score is driven entirely by retained growth.",
            detail, "Your score is a growth measurement wearing a "
            "product-match label. If growth coupling is what you intended, "
            "say so explicitly and score it directly; if not, constrain the "
            "product route so the match term varies.", ev, cites)
    if p_inert and g_inert:
        return CheckResult(
            "D5", "growth confound", WARN,
            "Neither the product-match nor the growth term varies "
            "meaningfully across candidates.",
            detail, "Nothing in the score responds to the intervention. See "
            "D1 and D2.", ev, cites)
    if not np.isnan(cg) and abs(cg) > 0.95 and (np.isnan(cp) or abs(cp) < 0.3):
        return CheckResult(
            "D5", "growth confound", WARN,
            "The score follows growth almost exactly and barely follows the "
            "product match.",
            detail, "Check this is the growth-coupled behaviour you intended "
            "rather than the product term being inert.", ev, cites)
    if g_inert and not p_inert:
        return CheckResult(
            "D5", "growth confound", PASS,
            "The score follows the product-match term; growth is constant "
            "across these candidates, so it is not confounding the ranking.",
            detail, evidence=ev, citations=cites)
    return CheckResult(
        "D5", "growth confound", PASS,
        "Both the product-match and growth terms vary and contribute to the "
        "score.", detail, evidence=ev, citations=cites)


# ---------------------------------------------------------------------------
# D6 -- parameter sensitivity                                        [KNOWN]
# ---------------------------------------------------------------------------

def check_parameter_sensitivity(evaluator, target_profile, scores,
                                 probe_combos=None) -> CheckResult:
    """Do the scoring parameters actually change anything?

    Ordinary sensitivity analysis. If scores are unchanged when a key
    parameter moves, that parameter's constraint is not binding, which often
    means it is not doing the job you think it is.
    """
    import dataclasses
    cfg = getattr(evaluator, "config", None)
    cites = ["standard sensitivity analysis; "
             "Bernstein et al. 2021 (doi:10.1186/s13059-021-02289-z)"]
    if cfg is None or not dataclasses.is_dataclass(cfg):
        return CheckResult("D6", "parameter sensitivity", SKIP,
                           "Needs a dataclass config to perturb.",
                           citations=cites)
    # NOTE: this rebuilds the evaluator once per parameter value, which for a
    # COBRA evaluator means reloading the model each time. It is the slowest
    # check in the suite by a wide margin. Pass fewer probe_combos, or skip
    # this check, if that cost matters.
    probe_combos = probe_combos or list(scores)[:4]
    if not probe_combos:
        return CheckResult("D6", "parameter sensitivity", SKIP,
                           "No candidates to probe.", citations=cites)

    EvalCls = type(evaluator)
    sweeps = {"ratio_match_alpha": [0.2, 0.8],
              "hard_min_growth_fraction": [0.05, 0.3]}
    rows, inert = [], []
    for param, values in sweeps.items():
        if not hasattr(cfg, param):
            continue
        base = [float(evaluator.evaluate(c, target_profile))
                for c in probe_combos]
        deltas = []
        for v in values:
            try:
                alt = EvalCls(dataclasses.replace(cfg, **{param: v}))
                got = [float(alt.evaluate(c, target_profile))
                       for c in probe_combos]
            except Exception as exc:  # noqa: BLE001
                rows.append(f"{param}={v}: could not evaluate ({exc})")
                continue
            d = float(np.max(np.abs(np.array(got) - np.array(base))))
            deltas.append(d)
            rows.append(f"{param}={v:<5} max score change {d:.3e}")
        if deltas and max(deltas) <= SOLVER_NOISE_FLOOR:
            inert.append(param)

    if not rows:
        return CheckResult("D6", "parameter sensitivity", SKIP,
                           "No recognised parameters to sweep.",
                           citations=cites)
    detail = "\n".join(rows)
    ev = {"inert_parameters": inert}
    if inert:
        return CheckResult(
            "D6", "parameter sensitivity", WARN,
            f"Changing {', '.join(inert)} does not move any score.",
            detail, "That constraint is not binding. Either it is redundant, "
            "or the solution is pinned somewhere else and the parameter can "
            "never take effect.", ev, cites)
    return CheckResult(
        "D6", "parameter sensitivity", PASS,
        "Scores respond to changes in the scoring parameters.",
        detail, evidence=ev, citations=cites)


# ---------------------------------------------------------------------------
# runner
# ---------------------------------------------------------------------------

def run_diagnostics(evaluator, target_profile, candidate_combos=None,
                     max_order=2, cap=40, verbose=True) -> DiagnosticReport:
    """Run every check and return a report.

    `evaluator` needs `.evaluate(combo, target_profile) -> float`. Checks use
    more when it is there: `.config`, `.is_valid()`, `.diagnose()`, `._model`.
    Anything missing makes the relevant check SKIP rather than fail.
    """
    if candidate_combos is None:
        candidate_combos = _default_candidates(
            evaluator, target_profile, max_order=max_order, cap=cap)
    if verbose:
        print(f"[diagnostics] scoring {len(candidate_combos)} candidates ...")
    scores = _score_all(evaluator, candidate_combos, target_profile)

    results = [
        check_null_intervention_parity(evaluator, target_profile, scores),
        check_landscape_discrimination(evaluator, target_profile, scores),
        check_scalarization_degeneracy(evaluator, target_profile),
        check_free_production(evaluator, target_profile),
        check_growth_confound(evaluator, target_profile, scores),
        check_parameter_sensitivity(evaluator, target_profile, scores),
    ]
    return DiagnosticReport(results=results, target_profile=dict(target_profile),
                            n_candidates_tested=len(candidate_combos))

Writing rl_strain_design/diagnostics.py


In [9]:
%%writefile rl_strain_design/proposals.py
"""
proposals.py — wet-lab proposal sheets, gated on self-diagnosis
==================================================================
THE DESIGN IDEA

Every strain-design tool we surveyed -- OptKnock, OptGene, OptForce,
RobustKnock, cameo, MEWpy, OptFlux, StrainDesign -- will hand you a ranked
list of designs whether or not its own scoring landscape can support one.
None of them check first, and none refuse.

This module refuses. It runs diagnostics.py, and if a blocking check fails
it emits a "no proposals" sheet that explains what is wrong and what to fix
instead of a ranking that looks authoritative and means nothing. A dry lab
handing a wet lab three weeks of cloning work has an obligation not to do
that on the strength of a flat score.

WET-LAB ASSUMPTIONS (stated because they were not confirmed)

The sheets below assume a standard yeast workflow: knockouts built by
homologous recombination with selectable markers, and fatty-acid profile
read by GC-MS of derivatised total lipid extract. If the wet lab's actual
capability differs -- different profiling method, limited strain
construction throughput, no knockout capability yet -- the "how to test"
and "effort" fields need rewriting. They are the dry lab's guess, not the
wet lab's commitment.

CONFIDENCE TIERS

  A  model-discriminated and mechanistically expected
     The score separates this design from the alternatives, and the
     direction of the effect matches the known role of the genes.
  B  model-discriminated, mechanism unclear
     The score separates it, but we cannot say why from first principles.
     Worth testing, but a negative result is uninformative about the model.
  C  not discriminated
     Inside the noise of the scoring setup. Listed for completeness only.
     Do not spend bench time on these on our recommendation.

Tier A is still a model prediction, not evidence. Nothing here has been
validated against wet-lab data.
"""

from dataclasses import dataclass, field
import csv

from .diagnostics import (
    run_diagnostics, DiagnosticReport,
    DETECTABLE_RELATIVE_MARGIN, MEASURABLE_RELATIVE_MARGIN,
)


@dataclass
class Proposal:
    rank: int
    genes: tuple
    score: float
    margin_over_null: float
    tier: str
    rationale: str
    expected_signature: str
    falsified_if: str
    effort: str

    def as_row(self) -> dict:
        return {
            "rank": self.rank,
            "knockouts": " ".join(self.genes) if self.genes else "(none)",
            "n_genes": len(self.genes),
            "score": f"{self.score:.6f}",
            "margin_over_no_edit": f"{self.margin_over_null:.3e}",
            "confidence_tier": self.tier,
            "why": self.rationale,
            "expected_GCMS_signature": self.expected_signature,
            "model_falsified_if": self.falsified_if,
            "construction_effort": self.effort,
        }


@dataclass
class ProposalSheet:
    target_profile: dict
    diagnostics: DiagnosticReport
    proposals: list = field(default_factory=list)
    gated: bool = False
    gate_reason: str = ""
    assumptions: list = field(default_factory=list)

    def to_text(self) -> str:
        w = 78
        L = ["=" * w, "WET-LAB PROPOSAL SHEET", "=" * w,
             f"Target profile: " +
             ", ".join(f"{k}={v}" for k, v in self.target_profile.items()), ""]
        if self.gated:
            L += ["*** NO PROPOSALS ISSUED ***", "",
                  "The scoring setup did not pass its own diagnostics, so any",
                  "ranking it produced would be an artefact. Rather than hand",
                  "the wet lab a list we cannot stand behind, this sheet is",
                  "deliberately empty.", "", "Reason:", ""]
            for ln in self.gate_reason.split("\n"):
                L.append(f"  {ln}")
            L += ["", "Fix the blocking items in the diagnostic report, then",
                  "re-run. The diagnostic report is attached below.", ""]
        else:
            L += [f"{len(self.proposals)} proposal(s), highest confidence "
                  "first.", ""]
            for p in self.proposals:
                L += [f"--- Rank {p.rank}  [tier {p.tier}]  "
                      f"{' '.join(p.genes) if p.genes else '(none)'}",
                      f"    score            : {p.score:.6f}  "
                      f"(margin over no-edit {p.margin_over_null:.3e})",
                      f"    why              : {p.rationale}",
                      f"    expect to see    : {p.expected_signature}",
                      f"    model wrong if   : {p.falsified_if}",
                      f"    build effort     : {p.effort}", ""]
        if self.assumptions:
            L += ["-" * w, "ASSUMPTIONS THE DRY LAB MADE (please correct)", ""]
            L += [f"  - {a}" for a in self.assumptions]
            L.append("")
        L += ["-" * w, "DIAGNOSTIC REPORT", ""]
        L.append(self.diagnostics.to_text())
        return "\n".join(L)

    def to_csv(self, path: str) -> str:
        cols = ["rank", "knockouts", "n_genes", "score", "margin_over_no_edit",
                "confidence_tier", "why", "expected_GCMS_signature",
                "model_falsified_if", "construction_effort"]
        with open(path, "w", newline="") as f:
            wr = csv.DictWriter(f, fieldnames=cols)
            wr.writeheader()
            if self.gated:
                wr.writerow({c: "" for c in cols} | {
                    "knockouts": "NO PROPOSALS ISSUED",
                    "why": self.gate_reason.replace("\n", " ")})
            for p in self.proposals:
                wr.writerow(p.as_row())
        return path


# ---------------------------------------------------------------------------
# rationale / expectation text
# ---------------------------------------------------------------------------

# Plain-language roles, so a sheet reads the same to a wet-lab member as to a
# modeller. Keys are gene names; anything absent falls back to a neutral line.
GENE_ROLES = {
    "POX1": ("acyl-CoA oxidase, first committed step of peroxisomal "
             "beta-oxidation", "less fatty-acid breakdown"),
    "PXA1": ("peroxisomal ABC transporter subunit importing fatty acids for "
             "breakdown", "less fatty acid entering breakdown"),
    "PXA2": ("the other half of that transporter",
             "less fatty acid entering breakdown"),
    "FAA1": ("acyl-CoA synthetase that re-activates free fatty acids",
             "more free fatty acid left unactivated and available for export"),
    "FAA4": ("second acyl-CoA synthetase with longer-chain preference",
             "more free fatty acid left unactivated"),
    "DGA1": ("diacylglycerol acyltransferase, makes storage triacylglycerol",
             "less carbon diverted into storage lipid"),
    "LRO1": ("phospholipid-diacylglycerol acyltransferase, also makes storage "
             "triacylglycerol", "less carbon diverted into storage lipid"),
    "ARE1": ("sterol acyltransferase making steryl esters",
             "less carbon into sterol-linked storage"),
    "ARE2": ("the main sterol acyltransferase",
             "less carbon into sterol-linked storage"),
    "OLE1": ("delta-9 desaturase, the only route to unsaturated fatty acids",
             "loss of unsaturated species, which is lethal without supplement"),
}


def _rationale(genes) -> str:
    if not genes:
        return "No edit; included as the baseline comparison."
    parts = []
    for g in genes:
        role, effect = GENE_ROLES.get(
            g, (f"{g} (role not annotated in this config)", "an unknown effect"))
        parts.append(f"{g} is {role}, so deleting it should give {effect}")
    return "; ".join(parts) + "."


def _top_products(target_profile, n=1):
    """Highest-weighted products in the target, or [] for an empty target."""
    return [k for k, _ in
            sorted(target_profile.items(), key=lambda kv: -kv[1])[:n]]


def _signature(genes, target_profile) -> str:
    if not genes:
        return "The unedited profile, by definition."
    top = _top_products(target_profile, 2)
    if not top:
        return ("No target products were specified, so no expected signature "
                "can be stated.")
    want = " and ".join(top)
    return (f"A shift in the GC-MS profile toward {want} relative to the "
            "parent strain, measured as a fraction of total fatty-acid peak "
            "area, not absolute titre.")


def _falsified(genes, target_profile) -> str:
    if not genes:
        return "Not applicable."
    top = _top_products(target_profile, 1)
    if not top:
        return ("No target products were specified, so no falsification "
                "criterion can be stated.")
    top = top[0]
    return (f"The {top} fraction does not increase relative to the parent, or "
            "total fatty-acid output collapses while the ratio shifts, which "
            "would mean the model traded away yield that it should have kept.")


def _effort(genes) -> str:
    n = len(genes)
    if n == 0:
        return "None; this is the parent strain."
    if n == 1:
        return "One marker-based deletion; standard."
    if n == 2:
        return "Two sequential deletions; needs two markers or marker recycling."
    return (f"{n} sequential deletions; consider whether the marker budget and "
            "timeline justify it before starting.")


# ---------------------------------------------------------------------------
# builder
# ---------------------------------------------------------------------------

def build_proposal_sheet(evaluator, target_profile, candidate_combos,
                          top_n=5, diagnostics_report=None,
                          extra_assumptions=None,
                          mechanism_expected=None) -> ProposalSheet:
    """Score candidates, run diagnostics, and either rank or refuse.

    `candidate_combos` is whatever your search produced (or an enumeration).
    `mechanism_expected` is an optional set of gene names whose direction of
    effect you are willing to defend mechanistically; combos drawn entirely
    from it can reach tier A. Leave it None to let every discriminated combo
    be tier B, which is the conservative default.
    """
    report = diagnostics_report or run_diagnostics(
        evaluator, target_profile,
        candidate_combos=[tuple(c) for c in candidate_combos],
        verbose=False)

    assumptions = [
        "Knockouts are built by standard marker-based homologous "
        "recombination in the project's parent strain.",
        "Fatty-acid profile is read by GC-MS of derivatised total lipid "
        "extract, reported as fraction of total peak area.",
        "The wet lab's actual throughput and profiling method were not "
        "confirmed with them; effort and test columns are the dry lab's "
        "estimate.",
        "Point mutations in the project's mutation table are NOT applied by "
        "this evaluator, so these predictions describe knockout effects only.",
    ] + list(extra_assumptions or [])

    if not report.usable:
        reasons = []
        for r in report.blocking_failures:
            reasons.append(f"{r.check_id} {r.name}: {r.headline}")
            if r.fix:
                reasons.append(f"  -> {r.fix}")
        return ProposalSheet(
            target_profile=dict(target_profile), diagnostics=report,
            proposals=[], gated=True, gate_reason="\n".join(reasons),
            assumptions=assumptions)

    null = float(evaluator.evaluate((), target_profile))
    scored = sorted(
        ((tuple(c), float(evaluator.evaluate(tuple(c), target_profile)))
         for c in candidate_combos), key=lambda kv: -kv[1])

    proposals = []
    scale = max(abs(null), 1e-12)
    for i, (genes, score) in enumerate(scored[:top_n], start=1):
        margin = score - null
        rel = margin / scale
        # Tiers are set by whether an experiment could resolve the margin, not
        # by whether the solver could. See diagnostics.py on why.
        if rel < DETECTABLE_RELATIVE_MARGIN:
            tier = "C"
        elif rel < MEASURABLE_RELATIVE_MARGIN:
            tier = "B"
        elif mechanism_expected and set(genes) <= set(mechanism_expected):
            tier = "A"
        else:
            tier = "B"
        proposals.append(Proposal(
            rank=i, genes=genes, score=score, margin_over_null=margin,
            tier=tier, rationale=_rationale(genes),
            expected_signature=_signature(genes, target_profile),
            falsified_if=_falsified(genes, target_profile),
            effort=_effort(genes)))

    return ProposalSheet(target_profile=dict(target_profile),
                          diagnostics=report, proposals=proposals,
                          gated=False, assumptions=assumptions)

Writing rl_strain_design/proposals.py


In [10]:
%%writefile rl_strain_design/benchmark.py
"""
benchmark.py — does RL-guided search actually pay off, and when?
===================================================================
WHY THIS EXISTS

The honest objection to RL-guided knockout search at iGEM scale is that it
is unnecessary: with 10 candidate genes at order <= 3 there are only 175
combinations, so exhaustive enumeration is cheap and exact. Any claim that
a learned search helps has to show the regime where enumeration stops being
affordable, and has to show it on a landscape with real structure.

WHY THE BENCHMARK IS SYNTHETIC, STATED UP FRONT

We cannot demonstrate this on our own FBA landscape, because that landscape
is currently flat (see diagnostics.py D1/D2 -- doing nothing scores as well
as the best knockout). A search comparison on a flat landscape measures
nothing. So this module builds an explicit synthetic landscape with known
structure and known optima, and benchmarks search strategies on it.

What that does and does not establish:
  DOES    show how each search strategy's cost and solution quality scale
          with the number of candidate genes, on a landscape with
          main effects, pairwise epistasis and a growth penalty.
  DOES    locate the crossover where enumeration becomes impractical.
  DOES NOT show that RL finds better yeast strains. The landscape is
          synthetic. Transfer to a real GEM depends entirely on the real
          landscape having comparable structure, which for our project is
          not yet established.

Anyone reporting these numbers must carry that second block with them.
"""

from dataclasses import dataclass, field
import itertools
import random
import time

import numpy as np


# ---------------------------------------------------------------------------
# synthetic landscape
# ---------------------------------------------------------------------------

@dataclass
class SyntheticLandscape:
    """A deterministic fitness landscape over gene knockouts.

    Structure, chosen to mimic the qualitative features of a product-ratio
    FBA objective rather than any specific organism:

      * each gene has a main effect on the product vector (some genes push
        toward short chains, some toward long, some do nothing much)
      * each gene carries a growth cost
      * selected gene PAIRS interact (epistasis): their combined effect is
        not the sum of their parts, which is what makes the landscape worth
        searching rather than sorting
      * score = cosine(product vector, target) * growth penalty

    Every evaluation is counted, so search strategies can be compared on
    evaluation budget rather than wall-clock time.
    """
    n_genes: int
    n_products: int = 4
    seed: int = 0
    epistasis_pairs: int = 0          # 0 -> n_genes (one per gene, on average)
    epistasis_strength: float = 0.6
    growth_cost_scale: float = 0.08

    genes: list = field(default_factory=list, init=False)
    n_evals: int = field(default=0, init=False)

    def __post_init__(self):
        rng = np.random.default_rng(self.seed)
        self.genes = [f"G{i:02d}" for i in range(self.n_genes)]
        self._rng = rng
        # main effect of deleting each gene on the product vector
        self._main = rng.normal(0, 1, size=(self.n_genes, self.n_products))
        # growth cost of deleting each gene (always non-negative)
        self._cost = np.abs(rng.normal(0, 1, size=self.n_genes)) * \
            self.growth_cost_scale
        # baseline product vector of the unedited strain
        self._base = np.abs(rng.normal(1.0, 0.25, size=self.n_products))
        # pairwise interactions on a sparse set of pairs
        n_pairs = self.epistasis_pairs or self.n_genes
        all_pairs = list(itertools.combinations(range(self.n_genes), 2))
        rng.shuffle(all_pairs)
        self._pairs = {}
        for (i, j) in all_pairs[:min(n_pairs, len(all_pairs))]:
            self._pairs[(i, j)] = rng.normal(
                0, self.epistasis_strength, size=self.n_products)
        self._cache = {}

    # -- the evaluator interface the search code expects --------------------

    def evaluate(self, knockout_combo, target_profile: dict) -> float:
        key = (tuple(sorted(knockout_combo)),
               tuple(sorted(target_profile.items())))
        if key in self._cache:
            return self._cache[key]
        self.n_evals += 1

        idx = [self.genes.index(g) for g in knockout_combo if g in self.genes]
        vec = self._base.copy()
        for i in idx:
            vec = vec + self._main[i]
        for a, b in itertools.combinations(sorted(idx), 2):
            if (a, b) in self._pairs:
                vec = vec + self._pairs[(a, b)]
        vec = np.clip(vec, 0.0, None)

        growth = float(np.clip(1.0 - sum(self._cost[i] for i in idx), 0.0, 1.0))

        t = np.array([target_profile.get(p, 0.0)
                      for p in self.product_names()])
        if np.linalg.norm(vec) < 1e-12 or np.linalg.norm(t) < 1e-12:
            score = 0.0
        else:
            cos = float(np.dot(vec, t) /
                        (np.linalg.norm(vec) * np.linalg.norm(t)))
            # Hill-shaped growth penalty, same form as the FBA evaluator
            d = 1.0 - growth
            penalty = (d ** 2) / (0.5 ** 2 + d ** 2)
            score = cos * (1.0 - penalty)

        self._cache[key] = score
        return score

    def product_names(self):
        return [f"P{i}" for i in range(self.n_products)]

    def random_target(self, rng=None):
        rng = rng or self._rng
        w = rng.dirichlet(np.ones(self.n_products))
        return {p: round(float(x), 3)
                for p, x in zip(self.product_names(), w)}

    def reset_counter(self):
        self.n_evals = 0
        self._cache = {}


# ---------------------------------------------------------------------------
# search strategies, all on the same evaluation-budget footing
# ---------------------------------------------------------------------------

def search_exhaustive(landscape, target, max_order):
    """Every combination up to max_order. Exact, and the cost baseline."""
    best, best_score = None, -np.inf
    for order in range(1, max_order + 1):
        for combo in itertools.combinations(landscape.genes, order):
            s = landscape.evaluate(combo, target)
            if s > best_score:
                best, best_score = combo, s
    return best, best_score


def search_random(landscape, target, max_order, budget, seed=0):
    """Random valid combos, same evaluation budget as the learned search."""
    rng = random.Random(seed)
    best, best_score = None, -np.inf
    for _ in range(budget):
        order = rng.randint(1, max_order)
        combo = tuple(rng.sample(landscape.genes, order))
        s = landscape.evaluate(combo, target)
        if s > best_score:
            best, best_score = combo, s
    return best, best_score


def search_greedy(landscape, target, max_order):
    """Add the single best next gene each step. Cheap, and the strategy
    epistasis is specifically expected to defeat."""
    combo, best_score = (), landscape.evaluate((), target)
    for _ in range(max_order):
        cand, cand_score = None, best_score
        for g in landscape.genes:
            if g in combo:
                continue
            s = landscape.evaluate(combo + (g,), target)
            if s > cand_score:
                cand, cand_score = combo + (g,), s
        if cand is None:
            break
        combo, best_score = cand, cand_score
    return combo, best_score


def search_rl(landscape, target_sampler, eval_target, max_order,
               n_episodes=300, seed=0, lr=0.08, epsilon=0.25, gamma=0.9):
    """Linear Q-learning over (target profile, partial combo, next gene).

    Deliberately the same algorithm class as the project's own agent, written
    standalone here so the benchmark does not depend on a SearchConfig or a
    COBRA model.
    """
    rng = random.Random(seed)
    nprng = np.random.default_rng(seed)
    genes = landscape.genes
    n_p = landscape.n_products
    dim = n_p + 2 * len(genes) + 2
    w = np.zeros(dim)

    def phi(target, combo, action):
        tv = np.array([target.get(p, 0.0) for p in landscape.product_names()])
        a = np.array([1.0 if g == action else 0.0 for g in genes])
        c = np.array([1.0 if g in combo else 0.0 for g in genes])
        return np.concatenate([tv, a, c, [1.0], [len(combo) / max_order]])

    def q(target, combo, action):
        p = phi(target, combo, action)
        return float(w @ p), p

    eps = epsilon
    for _ in range(n_episodes):
        target = target_sampler(nprng)
        combo = ()
        prev = landscape.evaluate(combo, target)
        for step in range(max_order):
            avail = [g for g in genes if g not in combo]
            if not avail:
                break
            if rng.random() < eps:
                action = rng.choice(avail)
            else:
                action = max(avail, key=lambda g: q(target, combo, g)[0])
            nxt = combo + (action,)
            s = landscape.evaluate(nxt, target)
            reward = s - prev
            done = step == max_order - 1
            q_sa, p = q(target, combo, action)
            if done:
                tgt = reward
            else:
                rest = [g for g in genes if g not in nxt]
                tgt = reward + (gamma * max(q(target, nxt, g)[0] for g in rest)
                                if rest else 0.0)
            w += lr * (tgt - q_sa) * p
            combo, prev = nxt, s
        eps = max(0.02, eps * 0.995)

    # greedy rollout on the target we actually care about
    combo = ()
    best_score = landscape.evaluate(combo, eval_target)
    for _ in range(max_order):
        avail = [g for g in genes if g not in combo]
        if not avail:
            break
        action = max(avail, key=lambda g: q(eval_target, combo, g)[0])
        nxt = combo + (action,)
        s = landscape.evaluate(nxt, eval_target)
        if s > best_score:
            combo, best_score = nxt, s
        else:
            combo = nxt
            best_score = max(best_score, s)
    return combo, best_score


# ---------------------------------------------------------------------------
# the scaling experiment
# ---------------------------------------------------------------------------

@dataclass
class BenchRow:
    n_genes: int
    max_order: int
    strategy: str
    best_score: float
    frac_of_exhaustive: float
    n_evals: int
    seconds: float
    combo: tuple


def combination_count(n_genes, max_order):
    from math import comb
    return sum(comb(n_genes, k) for k in range(1, max_order + 1))


def run_scaling_benchmark(gene_counts=(10, 20, 30, 40, 50), max_order=3,
                           n_episodes=300, seed=0, fba_seconds_per_eval=0.30,
                           verbose=True):
    """Compare strategies as the candidate gene set grows.

    `fba_seconds_per_eval` converts evaluation counts into the wall-clock
    cost they would carry on a real genome-scale model. 0.30 s is measured
    from this project's own 3-stage lexicographic Yeast9 evaluator; set it to
    your own measured figure before quoting any projected time.
    """
    rows = []
    for n in gene_counts:
        land = SyntheticLandscape(n_genes=n, seed=seed)
        target = land.random_target(np.random.default_rng(seed + 1))
        total_combos = combination_count(n, max_order)

        # exhaustive, the exact reference
        land.reset_counter()
        t0 = time.time()
        ex_combo, ex_score = search_exhaustive(land, target, max_order)
        ex_evals, ex_sec = land.n_evals, time.time() - t0
        rows.append(BenchRow(n, max_order, "exhaustive", ex_score, 1.0,
                             ex_evals, ex_sec, ex_combo))

        # greedy
        land.reset_counter()
        t0 = time.time()
        g_combo, g_score = search_greedy(land, target, max_order)
        rows.append(BenchRow(n, max_order, "greedy", g_score,
                             g_score / ex_score if ex_score else 0.0,
                             land.n_evals, time.time() - t0, g_combo))

        # RL
        land.reset_counter()
        t0 = time.time()
        rl_combo, rl_score = search_rl(
            land, land.random_target, target, max_order,
            n_episodes=n_episodes, seed=seed)
        rl_evals, rl_sec = land.n_evals, time.time() - t0
        rows.append(BenchRow(n, max_order, "rl", rl_score,
                             rl_score / ex_score if ex_score else 0.0,
                             rl_evals, rl_sec, rl_combo))

        # random, matched to the RL evaluation budget -- the control that
        # separates "learning helped" from "sampling that many points helped"
        land.reset_counter()
        t0 = time.time()
        r_combo, r_score = search_random(land, target, max_order,
                                          budget=rl_evals, seed=seed)
        rows.append(BenchRow(n, max_order, "random_matched", r_score,
                             r_score / ex_score if ex_score else 0.0,
                             land.n_evals, time.time() - t0, r_combo))

        if verbose:
            print(f"n={n:3d}  combos={total_combos:>8,}  "
                  f"exhaustive={ex_score:.4f} ({ex_evals:,} evals)  "
                  f"rl={rl_score:.4f} ({rl_evals:,})  "
                  f"greedy={g_score:.4f}  random={r_score:.4f}", flush=True)
    return rows


def benchmark_table(rows, fba_seconds_per_eval=0.30) -> str:
    """Plain-text table, including what each evaluation count would cost on a
    real GEM at the given seconds-per-evaluation."""
    def hours(n):
        h = n * fba_seconds_per_eval / 3600.0
        if h < 1 / 60:
            return f"{n * fba_seconds_per_eval:.0f}s"
        if h < 1:
            return f"{h*60:.0f}min"
        return f"{h:.1f}h"

    L = [f"{'genes':>5} {'strategy':<16} {'score':>8} {'% of exact':>11} "
         f"{'evals':>9} {'projected FBA time':>19}",
         "-" * 74]
    for r in rows:
        L.append(f"{r.n_genes:>5} {r.strategy:<16} {r.best_score:>8.4f} "
                 f"{r.frac_of_exhaustive*100:>10.1f}% {r.n_evals:>9,} "
                 f"{hours(r.n_evals):>19}")
    L.append("-" * 74)
    L.append(f"projected time = evals x {fba_seconds_per_eval}s, this "
             "project's measured Yeast9 evaluator cost")
    return "\n".join(L)


def crossover_summary(rows, fba_seconds_per_eval=0.30,
                       practical_hours=8.0) -> str:
    """Where does enumeration stop being the right answer?

    `practical_hours` is the wall-clock budget past which exhaustive search
    stops being something a student can run between lab sessions.
    """
    by_n = {}
    for r in rows:
        by_n.setdefault(r.n_genes, {})[r.strategy] = r
    L = []
    crossover = None
    for n in sorted(by_n):
        ex = by_n[n].get("exhaustive")
        rl = by_n[n].get("rl")
        if not ex or not rl:
            continue
        ex_h = ex.n_evals * fba_seconds_per_eval / 3600.0
        if crossover is None and ex_h > practical_hours:
            crossover = n
        L.append(f"n={n:3d}: exhaustive needs {ex.n_evals:,} evaluations "
                 f"({ex_h:.1f} h on a real GEM); RL used {rl.n_evals:,} "
                 f"({rl.n_evals*fba_seconds_per_eval/3600.0:.1f} h) and "
                 f"reached {rl.frac_of_exhaustive*100:.1f}% of the exact "
                 "optimum")
    head = (f"Enumeration passes the {practical_hours:.0f}-hour practical "
            f"budget at about {crossover} candidate genes."
            if crossover else
            f"Enumeration stayed within the {practical_hours:.0f}-hour budget "
            "at every size tested, so a learned search is not needed in this "
            "range.")
    return head + "\n" + "\n".join(L)

Writing rl_strain_design/benchmark.py


In [11]:
%%writefile configs/yeast9_methyl_myristate.yaml
# yeast9_methyl_myristate.yaml
# =================================================================
# Real example config: iGEM IITB 2026 dry-lab biocoolant project.
# Engineering S. cerevisiae to shift fatty-acid output toward C14:0
# (myristic acid), the feedstock for methyl myristate, a biodegradable
# dielectric coolant for data centers.
#
# Every value below was validated against the actual Yeast9 GEM
# (yeast-GEM.xml, SysBioChalmers/yeast-GEM) during this project -- see
# `sources` at the bottom for where each number came from. Use this file
# as the template for a new organism/project: copy it, swap every value,
# and nothing in rl_strain_design/ needs to change.

name: igem-iitb-2026-methyl-myristate
description: >
  RL-guided knockout search for shifting S. cerevisiae fatty-acid output
  toward a target C14:0-rich profile, for methyl myristate biocoolant
  production.

# --- model / organism ---
model_path: yeast-GEM.xml
biomass_reaction_id: r_4041   # "biomass pseudoreaction" -- confirmed present
                               # in Yeast9. NOTE: part of the team pipeline
                               # (Bipra/Gururatna) uses r_2111 instead for the
                               # same purpose -- this is an unresolved
                               # cross-module conflict, not a typo. Confirm
                               # which one the team has settled on before
                               # trusting cross-module numeric comparisons.

# --- candidate gene space (Module 3 gene list) ---
candidate_genes:
  - POX1
  - PXA1
  - PXA2
  - FAA1
  - FAA4
  - DGA1
  - LRO1
  - ARE1
  - ARE2
  - OLE1

gene_to_model_id:   # common name -> Yeast9 systematic ORF ID (all verified)
  POX1: YGL205W
  PXA1: YPL147W
  PXA2: YKL188C
  FAA1: YOR317W
  FAA4: YMR246W
  DGA1: YOR245C
  LRO1: YNR008W
  ARE1: YCR048W
  ARE2: YNR019W
  OLE1: YGL055W

max_knockout_order: 3   # matches Module 3's order-3 cap

# --- target product(s): fatty-acid exchange reactions tracked ---
# Only FAs confirmed against a real Module 1 FA_METABOLITE_MAP entry are
# listed here. C16:1 (Gururatna's addition) was NOT in Sid's confirmed
# mapping and is deliberately left out rather than assumed equivalent --
# add it back in once someone confirms its exchange reaction ID.
exchange_reactions:
  "C14:0": r_2193
  "C16:0": r_1993
  "C18:0": r_2055
  "C18:1": r_2189

# --- search-space pruning rules (replicates Module 3's pre-FBA filters) ---
pre_fba_filters:
  # FAA1 + FAA4 + any third gene knocked out together is lethal.
  - type: mutual_exclusion_at_order
    genes: [FAA1, FAA4]
    min_order: 3

  # Don't knock out the only desaturase (OLE1) if the target profile still
  # wants any C18:1 (the unsaturated product OLE1 makes) -- self-contradictory.
  - type: conditional_exclusion
    gene: OLE1
    condition:
      target_profile_key: "C18:1"
      operator: ">"
      value: 0

  # Arhan (05/09, verbatim): "you only need to take care that we don't
  # immediately propose the knockdown of degradation in the first
  # iteration of the model run / we propose it in subsequent runs /
  # essentially i only want to ensure that 'knock down the degradation
  # pathway' does not go as a request to wet labs in the first iteration."
  # Gate defaults CLOSED (first-iteration behaviour). To run a
  # later-iteration search that's allowed to consider degradation
  # knockouts, pass open_gates={"allow_degradation_knockouts"} explicitly
  # to the agent/evaluator calls -- do not flip `default` to true here,
  # since that would silently change first-iteration behaviour for anyone
  # reusing this config file as-is.
  - type: gated_subset
    genes: [POX1, PXA1, PXA2]
    gate_name: allow_degradation_knockouts
    default: false

# --- scoring / objective shape ---
ratio_match_alpha: 0.5        # Stage 2 requires >= 50% of Stage 1's max total output
hard_min_growth_fraction: 0.01  # hard floor -- rejects genuinely dead strains
growth_penalty_K: 0.5          # growth-deficit at which the Hill penalty is half-maximal
growth_penalty_n: 2.0          # Hill coefficient (steepness)

sources:
  - "iGEM IITB 2026 dry-lab Modules 1-5 (team pipeline, see project docs)"
  - "SysBioChalmers/yeast-GEM (Yeast9 genome-scale model)"
  - "Ohkubo et al. 2024 (Hill-type toxicity term, adapted for the growth penalty)"
  - "Arhan Khade, WhatsApp 05/09: degradation-knockout iteration gate instruction"

Writing configs/yeast9_methyl_myristate.yaml


## Step 3 — load the model

The one number everything else is relative to is the wild-type growth rate.

In [12]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0,'.')
from cobra.io import read_sbml_model
from rl_strain_design import *
import pandas as pd, numpy as np

config = load_config('configs/yeast9_methyl_myristate.yaml')
model = read_sbml_model(config.model_path)
WT = model.optimize().objective_value
print(f'wild-type growth: {WT:.6f} /h')
print('tracked products:', list(config.exchange_reactions))
print('candidate genes :', config.candidate_genes)

wild-type growth: 0.081100 /h
tracked products: ['C14:0', 'C16:0', 'C18:0', 'C18:1']
candidate genes : ['POX1', 'PXA1', 'PXA2', 'FAA1', 'FAA4', 'DGA1', 'LRO1', 'ARE1', 'ARE2', 'OLE1']


## Step 4 — FEASIBILITY: the growth / yield frontier

**The central question.** We hold growth at a fixed fraction of normal, maximise C14:0 export, then minimise the other fatty acids so the purity figure is the best achievable. Repeated for each candidate knockout set.

Read the `C14:0 share` column first.

In [13]:
GROWTH_FRACTIONS = [0.01, 0.10, 0.25, 0.50, 0.75, 0.90]
COMBOS = [(), ('FAA1',), ('FAA1','FAA4'), ('DGA1','FAA1','LRO1'),
          ('POX1','FAA1'), ('POX1','PXA1','FAA1')]
C14 = config.exchange_reactions['C14:0']

def frontier(combo):
    rows=[]
    with model as m:
        for g in combo:
            mid = config.gene_to_model_id.get(g)
            if mid:
                try: m.genes.get_by_id(mid).knock_out()
                except KeyError: pass
        for gf in GROWTH_FRACTIONS:
            with m as mm:
                mm.reactions.get_by_id(config.biomass_reaction_id).lower_bound = gf*WT
                mm.objective = mm.reactions.get_by_id(C14); mm.objective.direction='max'
                c14 = mm.slim_optimize()
                if c14 is None or c14!=c14:
                    rows.append({'growth %WT': gf*100, 'C14:0 mmol/gDW/h': None, 'C14:0 share': None}); continue
                if c14 < 1e-9:
                    rows.append({'growth %WT': gf*100, 'C14:0 mmol/gDW/h': 0.0, 'C14:0 share': 0.0}); continue
                mm.reactions.get_by_id(C14).lower_bound = 0.99*c14
                others=[mm.reactions.get_by_id(r) for n,r in config.exchange_reactions.items() if n!='C14:0']
                mm.objective={r:1.0 for r in others}; mm.objective.direction='min'
                s=mm.optimize()
                fl={n: max(0.0,s.fluxes.get(r,0.0)) for n,r in config.exchange_reactions.items()}
                tot=sum(fl.values())
                rows.append({'growth %WT': gf*100, 'C14:0 mmol/gDW/h': round(fl['C14:0'],4),
                             'C14:0 share': round(fl['C14:0']/tot,4) if tot>1e-12 else 0.0})
    return rows

for cb in COMBOS:
    print('knockouts:', ' '.join(cb) or '(no edit)')
    display(pd.DataFrame(frontier(cb)))

knockouts: (no edit)


,growth %WT,C14:0 mmol/gDW/h,C14:0 share
0,1.0,0.2307,1.0
1,10.0,0.2108,1.0
2,25.0,0.1776,1.0
3,50.0,0.1192,1.0
4,75.0,0.0596,1.0
5,90.0,0.0238,1.0


knockouts: FAA1


,growth %WT,C14:0 mmol/gDW/h,C14:0 share
0,1.0,0.2307,1.0
1,10.0,0.2108,1.0
2,25.0,0.1776,1.0
3,50.0,0.1192,1.0
4,75.0,0.0596,1.0
5,90.0,0.0238,1.0


knockouts: FAA1 FAA4


...

,growth %WT,C14:0 mmol/gDW/h,C14:0 share
0,1.0,0.2307,1.0
1,10.0,0.2108,1.0
2,25.0,0.1776,1.0
3,50.0,0.1192,1.0
4,75.0,0.0596,1.0
5,90.0,0.0238,1.0


knockouts: POX1 FAA1


,growth %WT,C14:0 mmol/gDW/h,C14:0 share
0,1.0,0.0,0.0
1,10.0,0.0,0.0
2,25.0,0.0,0.0
3,50.0,0.0,0.0
4,75.0,0.0,0.0
5,90.0,0.0,0.0


knockouts: POX1 PXA1 FAA1


,growth %WT,C14:0 mmol/gDW/h,C14:0 share
0,1.0,0.0,0.0
1,10.0,0.0,0.0
2,25.0,0.0,0.0
3,50.0,0.0,0.0
4,75.0,0.0,0.0
5,90.0,0.0,0.0


### What that table says

The unedited strain reaches **100% C14:0 share** at every growth level, trading yield linearly against growth: about 0.21 mmol/gDW/h at 10% growth down to 0.024 at 90%.

No knockout beats it. `FAA1` and `DGA1+FAA1+LRO1` are identical, `FAA1+FAA4` is ~2% worse, and both **POX1** sets give exactly zero.

**Cross-check against earlier independent work in this project:** a separate run reported 0.209152 mmol/gDW/h at >=10% growth. This setup gives ~0.211. Two independently built pipelines landing within 0.8% is the strongest evidence either of them is right.

## Step 5 — WHY: which route does the model use?

If POX1 deletion takes production to zero, POX1 must be on the only route. Let's see it.

In [14]:
with model as m:
    m.reactions.get_by_id(config.biomass_reaction_id).lower_bound = 0.5*WT
    m.objective = m.reactions.get_by_id(C14); m.objective.direction='max'
    sol = m.optimize()
print(f'C14:0 export at 50% WT growth: {sol.fluxes[C14]:.4f} mmol/gDW/h\n')
rows=[{'reaction': r.id, 'flux': round(sol.fluxes.get(r.id,0.0),4), 'name': r.name[:58]}
      for r in model.reactions
      if abs(sol.fluxes.get(r.id,0.0))>0.01 and any(c=='p' for c in r.compartments)]
display(pd.DataFrame(rows).head(14))

C14:0 export at 50% WT growth: 0.1192 mmol/gDW/h



,reaction,flux,name
0,r_0102,0.1192,acetyl-CoA C-acyltransferase (myristoyl-CoA)
1,r_0123,0.1192,acyl-CoA oxidase (hexadecanoyl-CoA)
2,r_0256,0.0596,catalase
3,r_0301,0.1192,"citrate synthase, peroxisomal"
4,r_0402,0.1192,fatty-acid--CoA ligase (hexadecanoate)
5,r_0412,-0.1192,fatty-acid--CoA ligase (tetradecanoate)
6,r_0715,-0.1192,"malate dehydrogenase, peroxisomal"
7,r_1689,0.1192,citrate/malate antiport
8,r_1772,-0.1192,fatty acid transport
9,r_1774,0.1192,fatty acid transport


### The route

Every step carries the same flux as the export (0.1192), which means a single linear chain:

```
C16:0  --r_0402-->  hexadecanoyl-CoA
       --r_0123-->  acyl-CoA oxidase          <- this is POX1
       --r_2251-->  enoyl-CoA hydratase        one round of
       --r_0102-->  myristoyl-CoA (C14)        beta-oxidation
       --r_0412-->  free C14:0  -->  export
```

The model takes **C16:0 and chews one two-carbon unit off it.** It does not make C14:0 by terminating synthesis early.

Two consequences:

1. **POX1 deletion abolishes C14:0** because r_0123 is a POX1 reaction. The plan to knock down degradation to preserve C14:0 is, in this model, exactly backwards.
2. **The model has no reaction for early C14 release from fatty-acid synthase** — which is what FAS2 V1254W is for. So it cannot represent our actual intervention. The mutation table's placeholder reaction IDs are a symptom of that absence, not a missing lookup.

## Step 6 — is that route a modelling artefact?

One step runs a fatty-acid–CoA ligase backwards. That reaction is ATP-driven and strongly favours the forward direction, so a reversal is thermodynamically dubious. If production depends on it, the route is an artefact. Test: forbid the reversal and re-run.

In [15]:
REV = ('r_0412','r_2196','r_2201','r_0402')
def maxc14(label, forward_only=False, ko=(), gf=0.5):
    with model as m:
        for g in ko:
            mid=config.gene_to_model_id.get(g)
            if mid:
                try: m.genes.get_by_id(mid).knock_out()
                except KeyError: pass
        if forward_only:
            for rid in REV:
                try:
                    rx=m.reactions.get_by_id(rid)
                    if rx.lower_bound<0: rx.lower_bound=0
                except KeyError: pass
        m.reactions.get_by_id(config.biomass_reaction_id).lower_bound=gf*WT
        m.objective=m.reactions.get_by_id(C14); m.objective.direction='max'
        v=m.slim_optimize()
    v = 0.0 if (v is None or v!=v) else float(v)
    print(f'{label:<48} {v:>8.4f}')
    return v

print('Max C14:0 export at >=50% WT growth:')
base = maxc14('as shipped (ligases reversible)')
fwd  = maxc14('ligases constrained forward-only', forward_only=True)
maxc14('POX1 deleted', ko=('POX1',))
maxc14('POX1 deleted AND ligases forward-only', forward_only=True, ko=('POX1',))

from cobra.flux_analysis import loopless_solution
with model as m:
    m.reactions.get_by_id(config.biomass_reaction_id).lower_bound=0.5*WT
    m.objective=m.reactions.get_by_id(C14); m.objective.direction='max'
    s=m.optimize(); ll=loopless_solution(m)
print(f'\nstandard FBA {s.fluxes[C14]:.4f}  vs  loopless {ll.fluxes[C14]:.4f}')
print('\nVerdict:', 'artefact' if fwd < 0.01*base else 'the route is REAL, not a reversal artefact')

Max C14:0 export at >=50% WT growth:
as shipped (ligases reversible)                    0.1192
ligases constrained forward-only                   0.1167
POX1 deleted                                      -0.0000
POX1 deleted AND ligases forward-only              0.0000



standard FBA 0.1192  vs  loopless 0.1192

Verdict: the route is REAL, not a reversal artefact


**Result: the route survives.** Production drops only ~2% (0.1192 to 0.1167) when the reversal is forbidden, and loopless FBA returns the identical flux. The beta-oxidation dependency is a genuine property of the model, not a numerical trick.

## Step 7 — was the earlier profile-matching score trustworthy?

Before the feasibility work, the pipeline scored knockouts by how closely the predicted fatty-acid profile matched a target. Six automated checks on that scoring function.

In [16]:
target = {'C14:0':0.6, 'C16:0':0.2, 'C18:0':0.1, 'C18:1':0.1}
evaluator = CobraEvaluator(config)
report = run_diagnostics(evaluator, target, max_order=2, cap=20)
print(report.to_text())

Loading model for 'igem-iitb-2026-methyl-myristate' (yeast-GEM.xml)...


Loaded. WT growth rate: 0.081100
[diagnostics] scoring 20 candidates ...


Loading model for 'igem-iitb-2026-methyl-myristate' (yeast-GEM.xml)...


Loaded. WT growth rate: 0.081100


Loading model for 'igem-iitb-2026-methyl-myristate' (yeast-GEM.xml)...


...

Loading model for 'igem-iitb-2026-methyl-myristate' (yeast-GEM.xml)...


Loaded. WT growth rate: 0.081100


Loading model for 'igem-iitb-2026-methyl-myristate' (yeast-GEM.xml)...


Loaded. WT growth rate: 0.081100


SCORING-SETUP DIAGNOSTIC REPORT
Target profile    : C14:0=0.6, C16:0=0.2, C18:0=0.1, C18:1=0.1
Candidates tested : 20
Result            : 0 pass, 3 warn, 3 fail, 0 skipped

VERDICT: NOT USABLE -- a blocking check failed; rankings are meaningless

FAIL  D1  null-intervention parity [new]
      The best intervention beats doing nothing by only 0.010%, which is below what the assay could resolve, so the ranking is untestable even though the arithmetic is sound.
      no-intervention score : 0.202509
      best candidate        : ['FAA1', 'FAA4'] -> 0.202529
      margin                : 1.973e-05  (0.010% of baseline)
      FIX: The objective is essentially satisfiable without any edit. Constrain the route the product must take (e.g. require flux through the synthesis pathway), bound the product exchange realistically, or score product yield at fixed growth instead of profile shape.
      refs: StrainDesign docs note the wild type can be returned as a solution (doi:10.1093/bioinformatics/

### Two checks fail, and the verdict is *not usable*

- The best knockout beat doing nothing by **0.010%**. GC-MS replicate error is 1–5%, so that ordering is untestable.
- All 20 candidates fall inside one measurement-resolution band.

The threshold being compared against is the **assay's**, not the solver's. An earlier version of this check used solver precision (1 part in a million) and passed this same broken setup.

## Step 8 — the tool refuses to issue proposals

In [17]:
import itertools
cands = [c for o in (1,2) for c in itertools.combinations(config.candidate_genes, o)
         if evaluator.is_valid(frozenset(c), target)][:20]
sheet = build_proposal_sheet(evaluator, target, cands, top_n=5, diagnostics_report=report)
print('gated (refused to issue proposals):', sheet.gated)
print(sheet.to_text()[:1400])

gated (refused to issue proposals): True
WET-LAB PROPOSAL SHEET
Target profile: C14:0=0.6, C16:0=0.2, C18:0=0.1, C18:1=0.1

*** NO PROPOSALS ISSUED ***

The scoring setup did not pass its own diagnostics, so any
ranking it produced would be an artefact. Rather than hand
the wet lab a list we cannot stand behind, this sheet is
deliberately empty.

Reason:

  D1 null-intervention parity: The best intervention beats doing nothing by only 0.010%, which is below what the assay could resolve, so the ranking is untestable even though the arithmetic is sound.
    -> The objective is essentially satisfiable without any edit. Constrain the route the product must take (e.g. require flux through the synthesis pathway), bound the product exchange realistically, or score product yield at fixed growth instead of profile shape.
  D2 landscape discrimination: The solver separates these candidates into 2 levels, but all of them fall inside one measurement-resolution band, so no experiment could confirm 

No published strain-design tool we surveyed — OptKnock, OptGene, OptForce, RobustKnock, cameo, MEWpy, OptFlux, StrainDesign — checks its own scoring landscape before emitting designs, and none refuses. That refusal is the part of this work with the clearest claim to being new.

## Step 9 — is a learned search worth it? (no)

Three strategies on a synthetic landscape with known optima, gene interactions and a growth cost, 5 random seeds, 30 genes. Synthetic because our real landscape is flat, and comparing search strategies on a flat landscape measures nothing.

In [18]:
from rl_strain_design.benchmark import (SyntheticLandscape, search_exhaustive,
    search_greedy, search_random, search_rl)
rows=[]
for eps in (0.0, 1.0, 4.0):
    g,r,q = [],[],[]
    for seed in range(5):
        L=SyntheticLandscape(n_genes=30, seed=seed, epistasis_strength=eps, epistasis_pairs=200)
        t=L.random_target(np.random.default_rng(seed+100))
        L.reset_counter(); _,ex=search_exhaustive(L,t,3)
        L.reset_counter(); _,gs=search_greedy(L,t,3)
        L.reset_counter(); _,rs=search_rl(L,L.random_target,t,3,n_episodes=300,seed=seed); budget=L.n_evals
        L.reset_counter(); _,qs=search_random(L,t,3,budget=budget,seed=seed)
        g.append(gs/ex); r.append(rs/ex); q.append(qs/ex)
    rows.append({'gene interaction': eps,
                 'greedy': f'{np.mean(g)*100:.1f}% ±{np.std(g)*100:.0f}',
                 'reinforcement learning': f'{np.mean(r)*100:.1f}% ±{np.std(r)*100:.0f}',
                 'random (same budget)': f'{np.mean(q)*100:.1f}% ±{np.std(q)*100:.0f}'})
print('% of the exhaustive (exact) optimum found:')
display(pd.DataFrame(rows))

% of the exhaustive (exact) optimum found:


,gene interaction,greedy,reinforcement learning,random (same budget)
0,0.0,99.9% ±0,82.2% ±10,99.9% ±0
1,1.0,99.4% ±1,79.7% ±12,99.6% ±1
2,4.0,99.7% ±0,84.8% ±9,99.8% ±0


### Greedy wins, and random ties it

Greedy reaches 99%+ using about 60 evaluations. The RL agent reaches 80–85% using 1,204, and swings ±10% by seed. **`greedy_search()` is now the recommended default** and the RL agent is documented as not recommended.

**On brute force.** Brute force means: try every combination, score each, keep the best. For 10 genes up to 3 at a time that is 175 combinations — about a minute of FBA, and it is *exact*. We do not beat it and should not claim to. What brute force cannot do is tell you whether "best" means anything: it will hand you a ranked list of 175 options with a confident winner when all 175 are identical within measurement error. That is what the checks in Step 7 are for. Different question, not a better search.

That random search ties greedy is itself a finding: this landscape has many near-optimal answers, and where that holds, clever search buys almost nothing.

## Summary

| Question | Answer |
| --- | --- |
| Can the model reach our profile? | Yes — 100% pure C14:0, unedited, up to 90% of normal growth |
| At what yield? | 0.21 mmol/gDW/h at 10% growth; 0.12 at 50%; 0.024 at 90% |
| Do knockouts help? | No. None improve on wild type; POX1 sets abolish production |
| Why? | The model's only C14:0 route is beta-oxidative shortening of C16:0 |
| Is that robust? | Yes — survives forward-only ligase constraints and loopless FBA |
| Was the old score trustworthy? | No — it ranked by 0.01% differences, below assay resolution |
| Is RL better than brute force? | No. Brute force is exact and cheap at this scale; greedy is the practical choice |
| What is the probability of success? | FBA cannot answer that. It gives upper limits, not likelihoods |

### Recommended next steps

1. **Add a FAS early-release reaction to the model**, documented and agreed across the team. Until then the model cannot represent the FAS2 strategy and no knockout search over it is meaningful.
2. **Report the growth/yield frontier as the headline modelling result** — it is quantitative, independently cross-validated, and tells the wet lab what titre to expect.
3. **Resolve the biomass reaction conflict** (this uses `r_4041`; other team code uses `r_2111`). Growth numbers are not comparable across modules until it is settled.